# Klasik Görüntü İşleme ile Hücre Segmentasyonu, Sayımı ve Zamansal Analizi

## Bölüm 2 – Veri Setinin İndirilmesi ve Ön İncelemesi

In [ ]:
# Bölüm 2.1–2.2: Veri seti klasörlerini ve sekansları inceleme
from pathlib import Path

VERI = Path("data/Fluo-N2DH-GOWT1")

for klasor in sorted(VERI.iterdir()):
    tif_sayisi = len(list(klasor.rglob("*.tif")))
    print(f"{klasor.name:12s} -> {tif_sayisi} tif dosyası")

In [ ]:
# Bölüm 2.3: Görüntülerin boyutu, veri tipi ve piksel yoğunlukları
#
# Test sızıntısını önlemek için:
#   - Sekans 01 (geliştirme + doğrulama havuzu): boyut, tip ve yoğunluklar detaylı incelenir.
#   - Sekans 02 (test havuzu): yalnızca kare sayısı, boyut ve veri tipi kontrol edilir.
#     Yoğunluk/histogram bilgisine bakılmaz; yöntem seçimini etkilememesi için.
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

def goruntu_oku(yol):
    """TIFF dosyasını orijinal bit derinliğiyle okur.
    cv2.imread Türkçe karakterli yollarda çalışmadığı için
    dosya önce bayt olarak okunur, sonra cv2.imdecode ile çözülür."""
    veri = np.fromfile(str(yol), dtype=np.uint8)
    return cv2.imdecode(veri, cv2.IMREAD_UNCHANGED)

# --- Sekans 02: yalnızca biçim kontrolü ---
dosyalar_02 = sorted((VERI / "02").glob("t*.tif"))
bicimler_02 = {(goruntu_oku(d).shape, str(goruntu_oku(d).dtype)) for d in dosyalar_02}
print(f"Sekans 02: {len(dosyalar_02)} kare, (boyut, tip) = {bicimler_02}")

# --- Sekans 01: detaylı inceleme ---
satirlar = []
for dosya in sorted((VERI / "01").glob("t*.tif")):
    g = goruntu_oku(dosya)
    satirlar.append({
        "kare": int(dosya.stem[1:]),          # "t042" -> 42
        "boyut": f"{g.shape[0]}x{g.shape[1]}",
        "tip": str(g.dtype),
        "min": int(g.min()),
        "max": int(g.max()),
        "ortalama": g.mean(),
        "p99": np.percentile(g, 99),          # piksellerin %99'u bu değerin altında
        "sifir_orani": (g == 0).mean(),
    })

istatistik = pd.DataFrame(satirlar)
print(f"Sekans 01: {len(istatistik)} kare, boyutlar = {istatistik.boyut.unique()}, tipler = {istatistik.tip.unique()}")

# Sekans 01 sayısal özeti (kareler arası dağılım)
istatistik[["min", "max", "ortalama", "p99", "sifir_orani"]].describe().round(3)

In [ ]:
# Bölüm 2.3: Histogram özellikleri - Sekans 01'in TÜM karelerinin birleşik histogramı
toplam = np.zeros(256)
for dosya in sorted((VERI / "01").glob("t*.tif")):
    toplam += np.bincount(goruntu_oku(dosya).ravel(), minlength=256)
hist_01 = toplam / toplam.sum()               # oran (0-1) olarak normalize

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(np.arange(256), hist_01, label="Sekans 01 (92 kare)")
ax.set_yscale("log")                           # arka plan tepesi diğer değerleri ezmesin
ax.set_xlabel("Piksel yoğunluğu")
ax.set_ylabel("Piksel oranı (log)")
ax.set_title("Sekans 01 - tüm karelerin birleşik histogramı")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# Sayısal özet: yoğunluğun hangi aralıkta toplandığı
kum = np.cumsum(hist_01)
for oran in [0.50, 0.90, 0.99, 0.999]:
    print(f"Piksellerin %{oran*100:g}'i <= {np.searchsorted(kum, oran)}")

In [ ]:
# Bölüm 2.3: Sekans 01'de parlaklığın kare kare (zamanla) değişimi
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(istatistik.kare, istatistik.ortalama)
ax[1].plot(istatistik.kare, istatistik.p99)
ax[0].set_title("Sekans 01 - kare ortalama yoğunluğu")
ax[1].set_title("Sekans 01 - kare 99. yüzdelik yoğunluğu (parlak bölgeler)")
for a in ax:
    a.set_xlabel("Kare numarası")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 2.4: Hücrelerin şekil, boyut ve parlaklık farklılıkları
#
# 01_GT/SEG klasöründeki manuel segmentasyon maskeleri hücre sınırlarını hassas biçimde verir.
# (Bu maskeler her karede tüm hücreleri değil, yalnızca bir kısmını içerir; ama ölçüm için yeterli.)
# Her etiketli hücre için alan, eşdeğer çap, şekil ve ortalama parlaklık ölçülür.
from skimage.measure import regionprops_table

SEG_KLASOR = VERI / "01_GT" / "SEG"
hucre_satirlari = []
for maske_yolu in sorted(SEG_KLASOR.glob("man_seg*.tif")):
    kare_no = int(maske_yolu.stem.replace("man_seg", ""))     # "man_seg014" -> 14
    maske = goruntu_oku(maske_yolu)                            # her hücre farklı bir etiket (1, 2, ...)
    goruntu = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    olcum = regionprops_table(
        maske, intensity_image=goruntu,
        properties=["label", "area", "equivalent_diameter_area", "eccentricity",
                    "solidity", "intensity_mean", "intensity_min", "intensity_max"])
    tablo = pd.DataFrame(olcum)
    tablo["kare"] = kare_no
    hucre_satirlari.append(tablo)

hucreler = pd.concat(hucre_satirlari, ignore_index=True)
print(f"{hucreler.kare.nunique()} SEG karesinden toplam {len(hucreler)} hücre ölçüldü.\n")
print("eccentricity: 0 = tam daire, 1'e yaklaştıkça uzun/elips")
print("solidity    : 1 = tamamen dışbükey, düşük değer = girintili/düzensiz şekil\n")

# Ölçümlerin dağılımı
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
ax[0].hist(hucreler.area, bins=25);                      ax[0].set_title("Alan (piksel)")
ax[1].hist(hucreler.equivalent_diameter_area, bins=25);  ax[1].set_title("Eşdeğer çap (piksel)")
ax[2].hist(hucreler.eccentricity, bins=25);              ax[2].set_title("Eccentricity")
ax[3].hist(hucreler.intensity_mean, bins=25);            ax[3].set_title("Ortalama parlaklık")
for a in ax:
    a.set_ylabel("Hücre sayısı")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

hucreler[["area", "equivalent_diameter_area", "eccentricity", "solidity",
          "intensity_mean", "intensity_min", "intensity_max"]].describe().round(2)

In [ ]:
# Bölüm 2.5 (a): Olası güçlüklerin görsel incelenmesi
#
# Ham görüntüler çok karanlık olduğu için (piksellerin %90'ı <= 6) YALNIZCA GÖSTERİM AMACIYLA
# kontrast artırılır: [0, p99.9] aralığına kırpılır ve gamma=0.5 uygulanır (soluk bölgeler açılır).
# Bu işlem segmentasyonda kullanılmaz.
# Kare seçimi keyfi olmasın diye sekansın başı, ortası ve sonu alınır.
# Kırmızı çizgiler: 01_GT/TRA takip etiketleri (referansa göre hücrelerin yeri).
from skimage.segmentation import find_boundaries

def gosterim_icin(g, ust_yuzdelik=99.9, gamma=0.5):
    ust = np.percentile(g, ust_yuzdelik)
    return (np.clip(g, 0, ust) / ust) ** gamma

ornek_kareler = [0, 45, 91]
fig, ax = plt.subplots(2, 3, figsize=(18, 12))
for i, kare_no in enumerate(ornek_kareler):
    g = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    tra = goruntu_oku(VERI / "01_GT" / "TRA" / f"man_track{kare_no:03d}.tif")

    ax[0, i].imshow(g, cmap="gray", vmin=0, vmax=255)
    ax[0, i].set_title(f"t{kare_no:03d} - ham görüntü (0-255)")

    renkli = np.dstack([gosterim_icin(g)] * 3)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    ax[1, i].imshow(renkli)
    ax[1, i].set_title(f"t{kare_no:03d} - gösterim için kontrast + TRA ({len(np.unique(tra)) - 1} hücre)")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 2.5 (b): Güçlüklerin sayısal olarak ölçülmesi (Sekans 01, 92 karenin tamamı)
#
# 1) Temas eden / çok yakın hücreler : TRA etiketleri 10 piksel genişletilince birleşen hücre sayısı
#    (TRA etiketleri gerçek hücreden biraz küçük olduğu için genişletme gerçek sınıra yaklaştırır)
# 2) Kontrast yetersizliği             : ortalama parlaklığı arka plana yakın olan hücrelerin oranı (Bölüm 2.4)
# 3) Gürültü                           : hücrelerin dışında kalan küçük parlak lekeler (>= 50 yoğunluk, < 100 piksel)
# 4) Düzensiz arka plan                : hücre dışı piksellerin 8x8 bloklardaki ortalaması
cekirdek = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))   # yarıçap 10 piksel
kare_satirlari = []
blok_haritalari = []
for kare_no in range(92):
    g = goruntu_oku(VERI / "01" / f"t{kare_no:03d}.tif")
    tra = goruntu_oku(VERI / "01_GT" / "TRA" / f"man_track{kare_no:03d}.tif")
    hucre_sayisi = len(np.unique(tra)) - 1

    genis = cv2.dilate((tra > 0).astype(np.uint8), cekirdek)
    bilesen_sayisi = cv2.connectedComponents(genis)[0] - 1          # 0 = arka plan

    hucre_disi = cv2.dilate(genis, cekirdek) == 0                   # hücrelerden emin olunacak kadar uzak bölge
    parlak = ((g >= 50) & hucre_disi).astype(np.uint8)
    n, _, stats, _ = cv2.connectedComponentsWithStats(parlak)
    kucuk_leke = int(np.sum(stats[1:, cv2.CC_STAT_AREA] < 100))

    bloklar = np.full((8, 8), np.nan)
    for r in range(8):
        for c in range(8):
            dilim = (slice(r * 128, (r + 1) * 128), slice(c * 128, (c + 1) * 128))
            if hucre_disi[dilim].sum() > 1000:
                bloklar[r, c] = g[dilim][hucre_disi[dilim]].mean()
    blok_haritalari.append(bloklar)

    kare_satirlari.append({"kare": kare_no, "hucre": hucre_sayisi,
                           "birlesen_hucre": hucre_sayisi - bilesen_sayisi,
                           "kucuk_parlak_leke": kucuk_leke,
                           "arka_plan_blok_min": np.nanmin(bloklar),
                           "arka_plan_blok_max": np.nanmax(bloklar)})

guclukler = pd.DataFrame(kare_satirlari)

print("1) Temas eden / çok yakın hücreler")
print(f"   Karelerin %{(guclukler.birlesen_hucre > 0).mean()*100:.0f}'inde en az bir birleşme var; "
      f"kare başına ortalama {guclukler.birlesen_hucre.mean():.2f}, en fazla {guclukler.birlesen_hucre.max()}")
print("2) Kontrast yetersizliği (SEG hücreleri)")
print(f"   Ortalama parlaklığı <= 6 olan (arka planın %90'ıyla aynı düzeyde) hücreler: "
      f"%{(hucreler.intensity_mean <= 6).mean()*100:.0f}; <= 10 olanlar: %{(hucreler.intensity_mean <= 10).mean()*100:.0f}")
print("3) Gürültü")
print(f"   Hücre dışında kare başına ortalama {guclukler.kucuk_parlak_leke.mean():.1f} küçük parlak leke "
      f"(en fazla {guclukler.kucuk_parlak_leke.max()})")
print("4) Düzensiz arka plan")
print(f"   Hücre dışı 8x8 blok ortalamaları: en düşük {guclukler.arka_plan_blok_min.mean():.2f}, "
      f"en yüksek {guclukler.arka_plan_blok_max.mean():.2f} (92 karenin ortalaması)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(guclukler.kare, guclukler.hucre, label="Referans hücre sayısı (TRA)")
ax[0].plot(guclukler.kare, guclukler.hucre - guclukler.birlesen_hucre, label="Genişletince ayrı kalan bileşen")
ax[0].set_xlabel("Kare numarası"); ax[0].set_title("Temas eden hücrelerin sayıma etkisi")
ax[0].legend(); ax[0].grid(alpha=0.3)
ort_harita = np.nanmean(blok_haritalari, axis=0)
im = ax[1].imshow(ort_harita, cmap="viridis")
ax[1].set_title("Hücre dışı arka plan - 8x8 blok ortalaması (92 kare)")
plt.colorbar(im, ax=ax[1])
plt.tight_layout()
plt.show()

## Bölüm 3 – Geliştirme, Doğrulama ve Test Görüntülerinin Seçilmesi

- Seed = öğrenci numarasının son 5 hanesi + 2026 = 29015 + 2026 = **31041**
- Sekans 01: 5 doğrulama + 10 geliştirme karesi (çakışmaz, aralarında en az 5 kare boşluk)
- Sekans 02: 3 test bloğu × ardışık 5 kare (çakışmaz, aralarında en az 5 kare boşluk)
- "En az 5 karelik boşluk": iki kare arasında en az 5 kare bulunmalı, yani kare numarası farkı ≥ 6.
- Koşullar sağlanmazsa aynı RNG akışından yeniden çekilir (deterministik tekrar).
- Seçim, algoritma geliştirilmeden önce yapılır ve sonradan değiştirilmez.

In [ ]:
# Bölüm 3: Rastgele ve tekrarlanabilir görüntü seçimi -> selected_frames.csv
# Bu hücre referans (GT) dosyalarını OKUMAZ; yalnızca kare numaralarını belirler.
OGRENCI_NO_SON5 = 29015
SEED = OGRENCI_NO_SON5 + 2026          # 31041

KARE_SAYISI = 92
DOGRULAMA_SAYISI = 5
GELISTIRME_SAYISI = 10
TEST_BLOK_SAYISI = 3
BLOK_UZUNLUGU = 5
MIN_BOSLUK = 5                         # iki kare arasında bulunması gereken en az kare sayısı
MAX_DENEME = 1000

def yeterince_uzak(a, b):
    """a ile b arasında en az MIN_BOSLUK kare varsa True (ör. 20 ile 26 -> arada 5 kare)."""
    return abs(a - b) - 1 >= MIN_BOSLUK

def sekans01_sec(rng):
    """Önce 5 doğrulama karesi, sonra onlardan yeterince uzak 10 geliştirme karesi seçer."""
    for deneme in range(1, MAX_DENEME + 1):
        dogrulama = np.sort(rng.choice(KARE_SAYISI, size=DOGRULAMA_SAYISI, replace=False))
        uygun = [k for k in range(KARE_SAYISI) if all(yeterince_uzak(k, d) for d in dogrulama)]
        if len(uygun) >= GELISTIRME_SAYISI:
            gelistirme = np.sort(rng.choice(uygun, size=GELISTIRME_SAYISI, replace=False))
            return dogrulama.tolist(), gelistirme.tolist(), deneme
    raise RuntimeError("Sekans 01 için koşulları sağlayan seçim bulunamadı")

def sekans02_sec(rng):
    """Birbirinden en az MIN_BOSLUK kare uzak 3 adet, 5 karelik ardışık blok seçer."""
    son_baslangic = KARE_SAYISI - BLOK_UZUNLUGU         # 87 -> blok 87..91
    for deneme in range(1, MAX_DENEME + 1):
        baslangiclar = np.sort(rng.choice(son_baslangic + 1, size=TEST_BLOK_SAYISI, replace=False))
        uygun_mu = all(baslangiclar[i + 1] - (baslangiclar[i] + BLOK_UZUNLUGU - 1) - 1 >= MIN_BOSLUK
                       for i in range(TEST_BLOK_SAYISI - 1))
        if uygun_mu:
            return [list(range(b, b + BLOK_UZUNLUGU)) for b in baslangiclar], deneme
    raise RuntimeError("Sekans 02 için koşulları sağlayan blok seçimi bulunamadı")

rng = np.random.default_rng(SEED)      # tek RNG akışı: aynı sıra -> her çalıştırmada aynı sonuç
dogrulama, gelistirme, deneme01 = sekans01_sec(rng)
test_bloklari, deneme02 = sekans02_sec(rng)

secim_satirlari = []
for k in gelistirme:
    secim_satirlari.append(("development", "01", k, ""))
for k in dogrulama:
    secim_satirlari.append(("validation", "01", k, ""))
for blok_no, kareler in enumerate(test_bloklari, start=1):
    for k in kareler:
        secim_satirlari.append(("test", "02", k, blok_no))

secim = pd.DataFrame(secim_satirlari, columns=["group", "sequence", "frame", "test_block"])
secim["image_file"] = [f"{s}/t{k:03d}.tif" for s, k in zip(secim.sequence, secim.frame)]
secim["reference_file"] = [f"{s}_GT/TRA/man_track{k:03d}.tif" for s, k in zip(secim.sequence, secim.frame)]
secim["seed"] = SEED

# Kural kontrolleri
assert len(secim) == 30
assert not set(gelistirme) & set(dogrulama)
assert all(yeterince_uzak(g, d) for g in gelistirme for d in dogrulama)
assert all((VERI / p).exists() for p in secim.image_file)     # yalnızca dosya var mı; içerik okunmaz

secim.to_csv("selected_frames.csv", index=False)

print(f"Seed: {SEED}")
print(f"Deneme sayısı -> Sekans 01: {deneme01}, Sekans 02: {deneme02}")
print(f"Doğrulama  (01): {dogrulama}")
print(f"Geliştirme (01): {gelistirme}")
for blok_no, kareler in enumerate(test_bloklari, start=1):
    print(f"Test blok {blok_no} (02): {kareler}")
print("Kaydedildi: selected_frames.csv")

# Seçimin zaman ekseninde görünümü
fig, ax = plt.subplots(figsize=(14, 2.2))
ax.scatter(gelistirme, [1] * len(gelistirme), marker="s", s=60, label="Geliştirme (01)")
ax.scatter(dogrulama, [1] * len(dogrulama), marker="D", s=60, label="Doğrulama (01)")
for kareler in test_bloklari:
    ax.scatter(kareler, [0] * len(kareler), marker="o", s=60, color="tab:red")
ax.scatter([], [], marker="o", color="tab:red", label="Test blokları (02)")
ax.set_yticks([0, 1]); ax.set_yticklabels(["Sekans 02", "Sekans 01"])
ax.set_xlim(-1, 92); ax.set_ylim(-0.6, 1.6)
ax.set_xlabel("Kare numarası"); ax.grid(alpha=0.3)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.45), ncol=3)
plt.show()

## Bölüm 4 – Referans Hücre Sayılarının Belirlenmesi

- Referans: `01_GT/TRA/man_trackXXX.tif` takip etiketleri. Her hücre farklı bir pozitif kimlikle (1, 2, 3, ...) işaretlidir; 0 arka plandır.
- Referans sayı = sıfır hariç birbirinden farklı kimliklerin sayısı.
- **Yalnızca geliştirme (10) ve doğrulama (5) karelerinin** referansları okunur.
- Test referansları (`02_GT/TRA`) nihai pipeline ve test tahminleri kilitlenip commit edilene kadar okunmaz; kod bunu bir kilit değişkeniyle engeller.

In [ ]:
# Bölüm 4: Geliştirme ve doğrulama karelerinin referans hücre sayıları
from scipy import ndimage

# Bölüm 12'de tahminler kilitlenip commit edilene kadar True kalacak
TEST_REFERANSLARI_KILITLI = True

def referans_oku(satir):
    """selected_frames.csv'deki bir satırın TRA etiket görüntüsünü okur.
    Test satırları kilit açılmadan okunamaz."""
    if satir.group == "test" and TEST_REFERANSLARI_KILITLI:
        raise PermissionError("Test referansları kilitli! Önce Bölüm 12'deki kilitleme tamamlanmalı.")
    return goruntu_oku(VERI / satir.reference_file)

ref_satirlari = []
for satir in secim[secim.group != "test"].itertuples():
    tra = referans_oku(satir)
    kimlikler = np.unique(tra)
    kimlikler = kimlikler[kimlikler != 0]                     # 0 = arka plan, sayılmaz

    # Belirsizlik kontrolleri (doküman madde 4):
    # - kenara değen hücre: görüntü sınırında kesilmiş olabilir
    # - parçalı etiket: aynı kimlik birden fazla kopuk parçadan oluşuyor mu
    kenar = np.unique(np.concatenate([tra[0], tra[-1], tra[:, 0], tra[:, -1]]))
    kenara_degen = int(np.sum(kenar != 0))
    parcali = sum(ndimage.label(tra == k)[1] > 1 for k in kimlikler)

    ref_satirlari.append({"group": satir.group, "frame": satir.frame,
                          "reference_count": len(kimlikler),
                          "kenara_degen": kenara_degen, "parcali_etiket": int(parcali)})

referanslar = pd.DataFrame(ref_satirlari)
print(referanslar.to_string(index=False))
print()
print(referanslar.groupby("group").reference_count.agg(["count", "mean", "min", "max"]).round(2))

In [ ]:
# Bölüm 4 (madde 5): Referans sayılarının görsel kontrolü - 15 geliştirme + doğrulama karesi
# Kontrast yalnızca gösterim için artırılır (Bölüm 2.5'teki gosterim_icin fonksiyonu).
# Kırmızı: TRA etiketlerinin sınırı. Her etiketin ortasına kimlik numarası yazılır.
gelistirme_dogrulama = secim[secim.group != "test"].reset_index(drop=True)
fig, ax = plt.subplots(3, 5, figsize=(25, 15.5))
for a, satir in zip(ax.ravel(), gelistirme_dogrulama.itertuples()):
    g = goruntu_oku(VERI / satir.image_file)
    tra = referans_oku(satir)
    renkli = np.dstack([gosterim_icin(g)] * 3)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    a.imshow(renkli)
    for k in np.unique(tra)[1:]:
        y, x = np.argwhere(tra == k).mean(axis=0)
        a.text(x, y, str(k), color="yellow", fontsize=7, ha="center", va="center")
    sayi = referanslar.loc[referanslar.frame == satir.frame, "reference_count"].item()
    grup = "geliştirme" if satir.group == "development" else "doğrulama"
    a.set_title(f"t{satir.frame:03d} ({grup}) - referans: {sayi} hücre")
    a.axis("off")
plt.tight_layout()
plt.show()

### Bölüm 4 – Görsel kontrol gözlemleri

1. **İşaretlenmemiş kenar hücresi:** Sol kenarda (x≈10, y≈320) görüntü sınırıyla kesilmiş soluk (ort. ≈8) bir hücre 15 karenin neredeyse hepsinde görülür ama TRA'da etiketi yoktur. Benzer biçimde t014, t078, t085'te üst kenarda kısmen görünen nesneler etiketsizdir. t020'de iç bölgede de etiketsiz soluk bir nesne vardır. Bu nesneleri bulan bir yöntem, referansa göre "fazla sayım" yapmış görünür.
2. **t020'deki küçük nesne (etiket 22) gerçek bir hücredir:** `man_track.txt`'e göre hücre 16, 16. karede bölünmüş; yavru hücreler 22 (17. kareden itibaren) ve 18'dir (21. kareden itibaren). Bölünmeden hemen sonra hücre küçük ve yoğundur; etiket alanı ≈250–400 pikselden başlayıp 50. kare civarında diğer hücrelerin boyutuna (≈1650) ulaşır. t085'teki 35 numaralı küçük nesne de hücre 23'ün yavrusudur.
3. **Sonuç:** Referans sayılar değiştirilmeden kullanılır. Kenar hücreleri ve yeni bölünmüş küçük hücreler hata analizinde ayrıca değerlendirilecektir. Minimum alan eşiği, yeni bölünmüş hücreleri silmeyecek kadar küçük, gürültü lekelerini (< 100 piksel) silecek kadar büyük seçilmelidir.

## Bölüm 5 – Başlangıç Yöntemi (Baseline)

Baseline, sonraki bütün yöntemlerin karşılaştırılacağı **basit referans yöntemdir**:

1. Gri seviyeli görüntüyü oku
2. Otsu eşikleme (tüm görüntü için tek, otomatik eşik)
3. Bağlı bileşen analizi (8-komşuluk)
4. Bileşenleri say (isteğe bağlı: sabit minimum alan eşiğinin altındakileri atla)
5. Sonucu görselleştir

Minimum alan eşiği **yalnızca geliştirme kareleri** üzerinde seçilir ve sonra sabitlenir. Baseline bundan sonra hiç değiştirilmez ve nihai yöntemle aynı test karelerinde karşılaştırılır.

**Metrikler** (gerçek = referans sayı, tahmin = yöntemin sayısı, hata = tahmin − gerçek):
- **MAE** = ortalama |hata| → ortalama kaç hücre yanılıyoruz
- **RMSE** = √(ortalama hata²) → büyük hataları daha çok cezalandırır
- **Bias** = ortalama hata → (+) fazla sayma, (−) eksik sayma eğilimi
- **Exact-count accuracy** = hatanın tam 0 olduğu karelerin oranı

In [ ]:
# Bölüm 5 (a): Baseline fonksiyonu, metrikler ve geliştirme kümesinde minimum alan seçimi
import time

def baseline_segment(g, min_alan=0):
    """Otsu eşikleme + 8-komşuluk bağlı bileşen analizi.
    Döndürür: (hücre sayısı, ikili maske, etiket görüntüsü, otsu eşiği)"""
    otsu_esik, maske = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, etiket, stats, _ = cv2.connectedComponentsWithStats(maske, connectivity=8)
    alanlar = stats[1:, cv2.CC_STAT_AREA]                  # 0 numaralı bileşen arka plan
    tutulan = np.flatnonzero(alanlar >= min_alan) + 1
    etiket = np.where(np.isin(etiket, tutulan), etiket, 0)
    return len(tutulan), maske, etiket, otsu_esik

def metrikler(tahmin, gercek):
    hata = np.asarray(tahmin) - np.asarray(gercek)
    return {"MAE": np.abs(hata).mean(),
            "RMSE": np.sqrt((hata ** 2).mean()),
            "Bias": hata.mean(),
            "Exact": (hata == 0).mean()}

# Geliştirme karelerini ve referanslarını bir kez belleğe al
gelistirme_kareleri = secim[secim.group == "development"].reset_index(drop=True)
dev_goruntuler = {s.frame: goruntu_oku(VERI / s.image_file) for s in gelistirme_kareleri.itertuples()}
dev_referans = referanslar.set_index("frame").reference_count

# Minimum alan eşiği taraması (yalnızca geliştirme kümesi)
ADAY_MIN_ALANLAR = [0, 10, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500]
tarama = []
for min_alan in ADAY_MIN_ALANLAR:
    tahmin = [baseline_segment(dev_goruntuler[k], min_alan)[0] for k in dev_goruntuler]
    gercek = [dev_referans[k] for k in dev_goruntuler]
    tarama.append({"min_alan": min_alan, **metrikler(tahmin, gercek)})
tarama = pd.DataFrame(tarama)
print("Geliştirme kümesi - minimum alan eşiğine göre baseline başarımı:")
print(tarama.round(2).to_string(index=False))

BASELINE_MIN_ALAN = int(tarama.loc[tarama.MAE.idxmin(), "min_alan"])
print(f"\nSeçilen baseline minimum alanı: {BASELINE_MIN_ALAN} piksel (en düşük geliştirme MAE)")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(tarama.min_alan, tarama.MAE, "o-", label="MAE")
ax.plot(tarama.min_alan, tarama.Bias, "s--", label="Bias")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xscale("symlog", linthresh=10)
ax.set_xlabel("Minimum nesne alanı (piksel, symlog)"); ax.set_ylabel("Hücre")
ax.set_title("Baseline - minimum alan eşiğinin etkisi (geliştirme kümesi)")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

# Sabitlenmiş baseline ile geliştirme kümesi sonuçları (kare bazında)
baseline_dev = []
for k, g in dev_goruntuler.items():
    t0 = time.perf_counter()
    sayi, maske, etiket, esik = baseline_segment(g, BASELINE_MIN_ALAN)
    sure = time.perf_counter() - t0
    baseline_dev.append({"frame": k, "reference": dev_referans[k], "baseline": sayi,
                         "hata": sayi - dev_referans[k], "otsu_esigi": esik,
                         "ham_bilesen": cv2.connectedComponents(maske, connectivity=8)[0] - 1,
                         "sure_ms": sure * 1000})
baseline_dev = pd.DataFrame(baseline_dev)
print(baseline_dev.round(1).to_string(index=False))
print({k: round(v, 3) for k, v in metrikler(baseline_dev.baseline, baseline_dev.reference).items()},
      f"| ortalama süre: {baseline_dev.sure_ms.mean():.1f} ms")

In [ ]:
# Bölüm 5 (b): Baseline segmentasyon çıktısının görselleştirilmesi
# Keyfi seçim olmasın diye: geliştirme kümesinde hatası EN BÜYÜK ve EN KÜÇÜK olan kareler gösterilir.
from skimage.color import label2rgb

en_kotu = baseline_dev.loc[baseline_dev.hata.abs().idxmax(), "frame"]
en_iyi = baseline_dev.loc[baseline_dev.hata.abs().idxmin(), "frame"]

fig, ax = plt.subplots(2, 3, figsize=(18, 12))
for satir_no, k in enumerate([en_kotu, en_iyi]):
    g = dev_goruntuler[k]
    tra = referans_oku(gelistirme_kareleri[gelistirme_kareleri.frame == k].iloc[0])
    sayi, maske, etiket, esik = baseline_segment(g, BASELINE_MIN_ALAN)

    renkli = np.dstack([gosterim_icin(g)] * 3)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    ax[satir_no, 0].imshow(renkli)
    ax[satir_no, 0].set_title(f"t{k:03d} - görüntü + referans (kırmızı): {dev_referans[k]} hücre")

    ax[satir_no, 1].imshow(maske, cmap="gray")
    ax[satir_no, 1].set_title(f"Otsu maskesi (eşik = {esik:.0f})")

    ax[satir_no, 2].imshow(label2rgb(etiket, bg_label=0))
    ax[satir_no, 2].set_title(f"Bileşenler (alan >= {BASELINE_MIN_ALAN}): {sayi} nesne, hata {sayi - dev_referans[k]:+d}")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

## Bölüm 6 – Yoğunluk Dönüşümlerinin Karşılaştırılması

**Kontrollü deney:** Yalnızca yoğunluk dönüşümü değiştirilir; geri kalan her şey baseline ile aynıdır (Otsu eşikleme → 8-komşuluk bağlı bileşen → minimum alan 50 piksel). Böylece sayımdaki değişim yalnızca dönüşüme bağlanabilir.

| ID | Dönüşüm | Parametre | Gerekçe |
|---|---|---|---|
| E01 | Orijinal (baseline) | – | Karşılaştırma noktası (zorunlu) |
| E02 | Contrast stretching | [0, p99.5] → [0, 255] | Kare maksimumu tek tük parlak piksellerle 195–255 arasında oynuyor (Bölüm 2.3); maksimum yerine yüzdelik kullanmak daha kararlı |
| E03 | Gamma | γ = 0.5 | Karanlık değerleri açar; soluk hücrelerin (ortanca parlaklık ≈14) sinyalini büyütür |
| E04 | Logaritmik | s = 255·log(1+r)/log(256) | Gamma'ya benzer ama karanlık uçta daha güçlü açar |
| E05 | Histogram eşitleme | – | Histogramı tüm aralığa yayar |
| E06 | CLAHE | clipLimit = 2, 8×8 blok | Yerel kontrast; 8×8 blok = 128 piksel ≈ 2 hücre çapı, düzensiz arka plana karşı |
| E07 | Histogram eşleme | Referans: geliştirme kümesinde p99'u ortanca olan kare | Kareler arası parlaklık farkını giderir; referans yalnızca geliştirme kümesinden |

In [ ]:
# Bölüm 6 (a): Yoğunluk dönüşümleri + kontrollü deney kayıtları
from skimage import exposure

# ---- Deney kayıt sistemi (Bölüm 10'daki experiments.csv bu listeden üretilecek) ----
DENEYLER = []

def dev_degerlendir(say_fonksiyonu):
    """Bir sayım fonksiyonunu 10 geliştirme karesinde çalıştırır: (tahminler, ortalama süre ms)."""
    tahminler, sureler = [], []
    for k, g in dev_goruntuler.items():
        t0 = time.perf_counter()
        tahminler.append(say_fonksiyonu(g))
        sureler.append(time.perf_counter() - t0)
    return tahminler, 1000 * np.mean(sureler)

def deney_ekle(deney_id, yogunluk, filtre, segmentasyon, morfoloji, parametreler, say_fonksiyonu):
    tahminler, sure_ms = dev_degerlendir(say_fonksiyonu)
    gercek = [dev_referans[k] for k in dev_goruntuler]
    kayit = {"deney_id": deney_id, "yogunluk": yogunluk, "filtre": filtre,
             "segmentasyon": segmentasyon, "morfoloji": morfoloji, "parametreler": parametreler,
             **metrikler(tahminler, gercek), "sure_ms": sure_ms,
             "tahminler": tahminler}
    DENEYLER[:] = [d for d in DENEYLER if d["deney_id"] != deney_id] + [kayit]   # tekrar çalıştırmada üzerine yaz
    return kayit

# ---- Yoğunluk dönüşümleri ----
def contrast_stretch(g, ust_yuzdelik=99.5):
    ust = np.percentile(g, ust_yuzdelik)
    return np.clip(g.astype(np.float32) * 255.0 / ust, 0, 255).astype(np.uint8)

def gamma_donusumu(g, gamma=0.5):
    return (255.0 * (g / 255.0) ** gamma).astype(np.uint8)

def log_donusumu(g):
    return (255.0 * np.log1p(g.astype(np.float32)) / np.log(256.0)).astype(np.uint8)

CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# Histogram eşleme referansı: geliştirme kümesinde p99 değeri ortanca olan kare (yalnızca geliştirme kümesinden)
dev_p99 = {k: np.percentile(g, 99) for k, g in dev_goruntuler.items()}
HM_REFERANS_KARE = sorted(dev_p99, key=dev_p99.get)[len(dev_p99) // 2]
HM_REFERANS = dev_goruntuler[HM_REFERANS_KARE]

def histogram_esleme(g):
    return exposure.match_histograms(g, HM_REFERANS).astype(np.uint8)

YOGUNLUK = {
    "E01": ("Orijinal",           "-",                                 lambda g: g),
    "E02": ("Contrast stretching", "[0, p99.5] -> [0, 255]",           contrast_stretch),
    "E03": ("Gamma",              "gamma=0.5",                         gamma_donusumu),
    "E04": ("Logaritmik",         "255*log(1+r)/log(256)",             log_donusumu),
    "E05": ("Histogram eşitleme", "-",                                 cv2.equalizeHist),
    "E06": ("CLAHE",              "clipLimit=2, 8x8 blok",             CLAHE.apply),
    "E07": ("Histogram eşleme",   f"referans t{HM_REFERANS_KARE:03d}", histogram_esleme),
}
print(f"Histogram eşleme referansı: t{HM_REFERANS_KARE:03d} (p99 = {dev_p99[HM_REFERANS_KARE]:.0f})\n")

for deney_id, (ad, param, donusum) in YOGUNLUK.items():
    deney_ekle(deney_id, ad, "-", "Otsu", "-", f"{param}; min_alan={BASELINE_MIN_ALAN}",
               lambda g, f=donusum: baseline_segment(f(g), BASELINE_MIN_ALAN)[0])

# Otsu eşiğinin dönüşüme göre değişimi (her karenin eşiği)
otsu_esikleri = pd.DataFrame({ad: [cv2.threshold(f(g), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]
                                   for g in dev_goruntuler.values()]
                              for ad, _, f in YOGUNLUK.values()}, index=list(dev_goruntuler)).T

tablo6 = pd.DataFrame(DENEYLER).set_index("deney_id")
tablo6["otsu_esigi_ort"] = otsu_esikleri.mean(axis=1).values
tablo6["hatalar"] = [[int(t - dev_referans[k]) for t, k in zip(d["tahminler"], dev_goruntuler)]
                     for d in DENEYLER]
print(tablo6[["yogunluk", "parametreler", "MAE", "RMSE", "Bias", "Exact", "otsu_esigi_ort", "sure_ms"]]
      .round(2).to_string())
print("\nKare bazında hata (tahmin - referans), kareler:", list(dev_goruntuler))
for deney_id, hatalar in tablo6["hatalar"].items():
    print(f"  {deney_id}: {hatalar}")

In [ ]:
# Bölüm 6 (b): İşlem öncesi/sonrası görüntüler, histogramlar ve Otsu maskeleri
# Gösterilen kare: baseline'ın en zorlandığı geliştirme karesi (Bölüm 5'te en büyük hata).
k = en_kotu
g = dev_goruntuler[k]
fig, ax = plt.subplots(3, len(YOGUNLUK), figsize=(4 * len(YOGUNLUK), 12))
for i, (deney_id, (ad, param, donusum)) in enumerate(YOGUNLUK.items()):
    d = donusum(g)
    sayi, maske, _, esik = baseline_segment(d, BASELINE_MIN_ALAN)

    ax[0, i].imshow(d, cmap="gray", vmin=0, vmax=255)       # dönüşümün kendisi görülsün diye kontrast artırılmaz
    ax[0, i].set_title(f"{deney_id} {ad}\n{param}", fontsize=10)
    ax[0, i].axis("off")

    ax[1, i].bar(np.arange(256), np.bincount(d.ravel(), minlength=256), width=1.0)
    ax[1, i].axvline(esik, color="red", lw=1, label=f"Otsu = {esik:.0f}")
    ax[1, i].set_yscale("log"); ax[1, i].set_xlim(0, 255)
    ax[1, i].legend(fontsize=8); ax[1, i].set_title("Histogram (log)", fontsize=10)

    ax[2, i].imshow(maske, cmap="gray")
    ax[2, i].set_title(f"Otsu maskesi: {sayi} nesne (ref {dev_referans[k]})", fontsize=10)
    ax[2, i].axis("off")
fig.suptitle(f"t{k:03d} - yoğunluk dönüşümleri (görüntü / histogram / Otsu maskesi)", fontsize=14)
plt.tight_layout()
plt.show()

## Bölüm 7 – Yumuşatma ve Keskinleştirme Filtrelerinin Karşılaştırılması

**Kontrollü deney:** Yalnızca filtre değişir; yoğunluk = orijinal, segmentasyon = Otsu, 8-komşuluk, min. alan 50 (baseline ile aynı). Karşılaştırma noktası filtresiz baseline'dır (E01).

| ID | Filtre | Kernel | Amaç |
|---|---|---|---|
| E08–E11 | Gaussian | 3×3, 5×5, 9×9, 15×15 | Gürültüyü bastırmak, soluk hücreleri tek parça yapmak. 3×3 ve 5×5 zorunlu; hücre çapı ≈60 piksel olduğu için daha büyük kernel'ler de denenir |
| E12–E14 | Median | 3×3, 5×5, 9×9 | Tuz-biber türü gürültüye karşı, kenarları Gaussian'a göre daha iyi korur |
| E15 | Laplacian keskinleştirme | [[0,−1,0],[−1,5,−1],[0,−1,0]] | Konvolüsyonla kenar vurgulama (zorunlu keskinleştirme) |
| E16 | Unsharp masking | g + 1.0·(g − Gauss_σ=2(g)) | Bulanık kopyayı çıkararak keskinleştirme |

**Yeni tanı ölçütleri:** Bölüm 5'te sayım hatalarının birbirini götürebildiğini gördük. Bu yüzden her deneyde bulunan nesneler TRA etiketleriyle eşleştirilir (yalnızca geliştirme kümesinde):
- **bulunan**: en az bir nesneyle örtüşen referans hücre sayısı
- **kaçırılan**: hiçbir nesneyle örtüşmeyen referans hücre sayısı
- **bölünme**: bir hücrenin birden fazla nesneye bölünmesinden gelen fazla sayım
- **birleşme**: birden fazla hücrenin tek nesnede birleşmesinden gelen eksik sayım
- **fazla**: hiçbir referans hücreyle örtüşmeyen nesne (gürültü, etiketsiz kenar hücresi)

In [ ]:
# Bölüm 7 (a): Filtreler + tanı ölçütleri (bulunan / kaçırılan / bölünme / birleşme / fazla)
from collections import Counter

dev_tra = {s.frame: referans_oku(s) for s in gelistirme_kareleri.itertuples()}

def tani_hesapla(etiket, tra):
    """Nesne etiketlerini TRA referans etiketleriyle örtüşmeye göre eşleştirir."""
    ortak = (etiket > 0) & (tra > 0)
    ciftler = np.unique(np.stack([etiket[ortak], tra[ortak]]), axis=1).T   # (nesne, hücre) çiftleri
    nesne_hucre, hucre_nesne = {}, {}
    for n, h in ciftler:
        nesne_hucre.setdefault(n, set()).add(h)
        hucre_nesne.setdefault(h, set()).add(n)
    nesneler = np.unique(etiket[etiket > 0])
    hucreler = np.unique(tra[tra > 0])
    return {"bulunan": len(hucre_nesne),
            "kacirilan": len(hucreler) - len(hucre_nesne),
            "bolunme": sum(len(v) - 1 for v in hucre_nesne.values()),
            "birlesme": sum(len(v) - 1 for v in nesne_hucre.values()),
            "fazla": len(nesneler) - len(nesne_hucre)}

def dev_tani(etiket_fonksiyonu):
    """Geliştirme kümesinde kare başına ortalama tanı ölçütleri."""
    toplam = Counter()
    for k, g in dev_goruntuler.items():
        toplam.update(tani_hesapla(etiket_fonksiyonu(g), dev_tra[k]))
    return {a: b / len(dev_goruntuler) for a, b in toplam.items()}

LAPLACIAN_KESKIN = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)

def unsharp_mask(g, sigma=2.0, miktar=1.0):
    bulanik = cv2.GaussianBlur(g, (0, 0), sigma)
    return cv2.addWeighted(g, 1 + miktar, bulanik, -miktar, 0)

FILTRELER = {
    "E01": ("Yok",                 "-",                  lambda g: g),
    "E08": ("Gaussian",            "3x3",                lambda g: cv2.GaussianBlur(g, (3, 3), 0)),
    "E09": ("Gaussian",            "5x5",                lambda g: cv2.GaussianBlur(g, (5, 5), 0)),
    "E10": ("Gaussian",            "9x9",                lambda g: cv2.GaussianBlur(g, (9, 9), 0)),
    "E11": ("Gaussian",            "15x15",              lambda g: cv2.GaussianBlur(g, (15, 15), 0)),
    "E12": ("Median",              "3x3",                lambda g: cv2.medianBlur(g, 3)),
    "E13": ("Median",              "5x5",                lambda g: cv2.medianBlur(g, 5)),
    "E14": ("Median",              "9x9",                lambda g: cv2.medianBlur(g, 9)),
    "E15": ("Laplacian keskinleştirme", "3x3 kernel (merkez 5)", lambda g: cv2.filter2D(g, -1, LAPLACIAN_KESKIN)),
    "E16": ("Unsharp masking",     "sigma=2, miktar=1",  unsharp_mask),
}

satirlar7 = []
for deney_id, (ad, param, filtre) in FILTRELER.items():
    if deney_id != "E01":      # E01 Bölüm 6'da zaten kaydedildi
        deney_ekle(deney_id, "Orijinal", f"{ad} {param}", "Otsu", "-", f"{param}; min_alan={BASELINE_MIN_ALAN}",
                   lambda g, f=filtre: baseline_segment(f(g), BASELINE_MIN_ALAN)[0])
    kayit = next(d for d in DENEYLER if d["deney_id"] == deney_id)
    tani = dev_tani(lambda g, f=filtre: baseline_segment(f(g), BASELINE_MIN_ALAN)[2])
    ham = np.mean([cv2.connectedComponents(baseline_segment(filtre(g))[1], connectivity=8)[0] - 1
                   for g in dev_goruntuler.values()])
    satirlar7.append({"deney_id": deney_id, "filtre": f"{ad} {param}", "MAE": kayit["MAE"],
                      "RMSE": kayit["RMSE"], "Bias": kayit["Bias"], "Exact": kayit["Exact"],
                      "ham_bilesen": ham, **tani, "sure_ms": kayit["sure_ms"]})

tablo7 = pd.DataFrame(satirlar7).set_index("deney_id")
print("Geliştirme kümesi (kare başına ortalamalar). Referans ortalaması:",
      round(np.mean(list(dev_referans[list(dev_goruntuler)])), 1), "hücre\n")
print(tablo7.round(2).to_string())

In [ ]:
# Bölüm 7 (b): Filtrelerin görsel karşılaştırması
# Kare: baseline'ın en zorlandığı geliştirme karesi. Ayrıntı görülsün diye, o karedeki
# EN SOLUK referans hücrenin çevresinden 300x300'lük bir bölge büyütülerek gösterilir.
k = en_kotu
g, tra = dev_goruntuler[k], dev_tra[k]
hucre_parlakligi = {h: g[tra == h].mean() for h in np.unique(tra)[1:]}
en_soluk = min(hucre_parlakligi, key=hucre_parlakligi.get)
cy, cx = np.argwhere(tra == en_soluk).mean(axis=0).astype(int)
y0, x0 = np.clip([cy - 150, cx - 150], 0, 1024 - 300)
kesit = (slice(y0, y0 + 300), slice(x0, x0 + 300))

fig, ax = plt.subplots(2, len(FILTRELER), figsize=(3.2 * len(FILTRELER), 7))
for i, (deney_id, (ad, param, filtre)) in enumerate(FILTRELER.items()):
    f = filtre(g)
    sayi, maske, etiket, esik = baseline_segment(f, BASELINE_MIN_ALAN)
    ax[0, i].imshow(gosterim_icin(f)[kesit], cmap="gray")
    ax[0, i].set_title(f"{deney_id} {ad}\n{param}", fontsize=9)
    renkli = label2rgb(etiket, bg_label=0)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    ax[1, i].imshow(renkli[kesit])
    ax[1, i].set_title(f"Otsu={esik:.0f} | tüm kare: {sayi} nesne", fontsize=9)
for a in ax.ravel():
    a.axis("off")
fig.suptitle(f"t{k:03d} - en soluk hücre (#{en_soluk}, ort. parlaklık {hucre_parlakligi[en_soluk]:.1f}) çevresi | "
             "üst: filtrelenmiş görüntü (gösterim için kontrastlı), alt: bulunan nesneler + referans (kırmızı)", fontsize=11)
plt.tight_layout()
plt.show()

## Bölüm 8 – Segmentasyon Yöntemlerinin Karşılaştırılması

Bölüm 7'nin bulgusu: kaçırılan ≈8 hücre filtreden bağımsız; darboğaz **tek global Otsu eşiğinin soluk hücrelerin üstünde kalması**. Bu bölümde yalnızca segmentasyon (eşikleme) yöntemi değiştirilir.

**Sabit tutulan ön işlem:** Orijinal yoğunluk + **Gaussian 15×15**. Gerekçe (Bölüm 7, geliştirme kümesi): en az parçalanma (bölünme 2,3), en az fazla nesne (0,1), en çok bulunan hücre (14,8) ve hız farkı yok. Sonrası: 8-komşuluk bağlı bileşen + min. alan 50 (baseline ile aynı).

| ID | Yöntem | Parametre seçimi |
|---|---|---|
| E11 | Otsu (global) | Otomatik; Bölüm 7'deki deney, karşılaştırma noktası |
| E17 | Sabit eşik (zorunlu) | T, geliştirme kümesinde 2–30 taranarak seçilir |
| E18 | Triangle (global) | Otomatik. Tek tepeli ve uzun kuyruklu histogramlar için tasarlanmış (floresan görüntülerde yaygın) |
| E19 | Li – minimum çapraz entropi (global) | Otomatik, parametresiz |
| E20 | Adaptif eşikleme (Gaussian ağırlıklı) | Blok boyutu ve C, geliştirme kümesinde taranır |
| E21 | Blok bazlı Otsu 8×8 | Her 128×128 blokta ayrı Otsu; hücre olmayan bloklar için alt sınır (taban) taranır |
| E22 | Blok bazlı Otsu 4×4 | Her 256×256 blokta ayrı Otsu; taban taranır |

Taranan bütün parametreler **yalnızca geliştirme kümesinde** seçilir.

In [ ]:
# Bölüm 8 (a): Segmentasyon yöntemleri - parametre taramaları (geliştirme kümesi)
from skimage.filters import threshold_li

def on_isleme_g15(g):
    return cv2.GaussianBlur(g, (15, 15), 0)

dev_g15 = {k: on_isleme_g15(g) for k, g in dev_goruntuler.items()}   # ön işlem bir kez yapılır

def maske_say(maske, min_alan=BASELINE_MIN_ALAN):
    """İkili maske -> (nesne sayısı, etiket görüntüsü). 8-komşuluk, min. alan filtresi."""
    n, etiket, stats, _ = cv2.connectedComponentsWithStats(maske.astype(np.uint8), connectivity=8)
    tutulan = np.flatnonzero(stats[1:, cv2.CC_STAT_AREA] >= min_alan) + 1
    return len(tutulan), np.where(np.isin(etiket, tutulan), etiket, 0)

# ---- Segmentasyon yöntemleri (girdi: ön işlenmiş görüntü, çıktı: ikili maske) ----
def sabit_esik(f, T):
    return f > T

def otsu_esik(f):
    return f > cv2.threshold(f, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]

def triangle_esik(f):
    return f > cv2.threshold(f, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)[0]

def li_esik(f):
    return f > threshold_li(f)

def adaptif_esik(f, blok, C):
    return cv2.adaptiveThreshold(f, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blok, C) > 0

def blok_otsu(f, n_blok, taban):
    """Görüntüyü n_blok x n_blok parçaya böler, her parçada ayrı Otsu eşiği hesaplar.
    Hücre içermeyen bloklarda Otsu gürültüyü ikiye böler; bunu önlemek için eşik en az 'taban' olur."""
    h, w = f.shape[0] // n_blok, f.shape[1] // n_blok
    maske = np.zeros(f.shape, bool)
    for r in range(n_blok):
        for c in range(n_blok):
            parca = f[r * h:(r + 1) * h, c * w:(c + 1) * w]
            esik = cv2.threshold(parca, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]
            maske[r * h:(r + 1) * h, c * w:(c + 1) * w] = parca > max(esik, taban)
    return maske

def dev_mae(segment_fonksiyonu):
    tahmin = [maske_say(segment_fonksiyonu(f))[0] for f in dev_g15.values()]
    return metrikler(tahmin, [dev_referans[k] for k in dev_g15])

# ---- Taramalar ----
tarama_sabit = pd.DataFrame([{"T": T, **dev_mae(lambda f, T=T: sabit_esik(f, T))} for T in range(2, 31)])
tarama_adaptif = pd.DataFrame([{"blok": b, "C": C, **dev_mae(lambda f, b=b, C=C: adaptif_esik(f, b, C))}
                               for b in [51, 101, 151, 201] for C in [-1, -2, -3, -4, -5, -6, -8, -10]])
tarama_blok = pd.DataFrame([{"n_blok": n, "taban": t, **dev_mae(lambda f, n=n, t=t: blok_otsu(f, n, t))}
                            for n in [4, 8] for t in [0, 2, 3, 4, 5, 6, 8, 10]])

EN_IYI_T = int(tarama_sabit.loc[tarama_sabit.MAE.idxmin(), "T"])
en_iyi_ad = tarama_adaptif.loc[tarama_adaptif.MAE.idxmin()]
ADAPTIF_BLOK, ADAPTIF_C = int(en_iyi_ad.blok), int(en_iyi_ad.C)
BLOK8_TABAN = int(tarama_blok[tarama_blok.n_blok == 8].set_index("taban").MAE.idxmin())
BLOK4_TABAN = int(tarama_blok[tarama_blok.n_blok == 4].set_index("taban").MAE.idxmin())
print(f"Seçilen parametreler -> sabit T={EN_IYI_T} | adaptif blok={ADAPTIF_BLOK}, C={ADAPTIF_C} | "
      f"blok Otsu 8x8 taban={BLOK8_TABAN} | blok Otsu 4x4 taban={BLOK4_TABAN}")

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(tarama_sabit["T"], tarama_sabit.MAE, "o-", label="MAE")
ax[0].plot(tarama_sabit["T"], tarama_sabit.Bias, "s--", label="Bias")
ax[0].axhline(0, color="gray", lw=0.8)
ax[0].set_xlabel("Sabit eşik T"); ax[0].set_title("Sabit eşik taraması"); ax[0].legend(); ax[0].grid(alpha=0.3)
pivot = tarama_adaptif.pivot(index="C", columns="blok", values="MAE")
ax[1].imshow(pivot, cmap="viridis_r")
ax[1].set_xticks(range(len(pivot.columns)), pivot.columns); ax[1].set_yticks(range(len(pivot.index)), pivot.index)
for (i, j), v in np.ndenumerate(pivot.values):
    ax[1].text(j, i, f"{v:.1f}", ha="center", va="center", color="white" if v > pivot.values.mean() else "black")
ax[1].set_xlabel("Blok boyutu"); ax[1].set_ylabel("C"); ax[1].set_title("Adaptif eşik - MAE")
for n in [4, 8]:
    t = tarama_blok[tarama_blok.n_blok == n]
    ax[2].plot(t.taban, t.MAE, "o-", label=f"{n}x{n} blok")
ax[2].set_yscale("log"); ax[2].set_xlabel("Taban eşik"); ax[2].set_title("Blok bazlı Otsu - MAE (log)")
ax[2].legend(); ax[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 8 (b): Seçilen parametrelerle deneyleri kaydet + tanı ölçütleri
SEGMENTASYONLAR = {
    "E11": ("Otsu (global)",     "otomatik",                             otsu_esik),
    "E17": ("Sabit eşik",        f"T={EN_IYI_T}",                        lambda f: sabit_esik(f, EN_IYI_T)),
    "E18": ("Triangle (global)", "otomatik",                             triangle_esik),
    "E19": ("Li (global)",       "otomatik",                             li_esik),
    "E20": ("Adaptif Gaussian",  f"blok={ADAPTIF_BLOK}, C={ADAPTIF_C}",  lambda f: adaptif_esik(f, ADAPTIF_BLOK, ADAPTIF_C)),
    "E21": ("Blok Otsu 8x8",     f"taban={BLOK8_TABAN}",                 lambda f: blok_otsu(f, 8, BLOK8_TABAN)),
    "E22": ("Blok Otsu 4x4",     f"taban={BLOK4_TABAN}",                 lambda f: blok_otsu(f, 4, BLOK4_TABAN)),
}

satirlar8 = []
for deney_id, (ad, param, segment) in SEGMENTASYONLAR.items():
    tam = lambda g, s=segment: maske_say(s(on_isleme_g15(g)))       # ön işlem dahil (süre ölçümü için)
    if deney_id != "E11":                                            # E11 Bölüm 7'de kaydedildi
        deney_ekle(deney_id, "Orijinal", "Gaussian 15x15", ad, "-", f"{param}; min_alan={BASELINE_MIN_ALAN}",
                   lambda g, t=tam: t(g)[0])
    kayit = next(d for d in DENEYLER if d["deney_id"] == deney_id)
    tani = dev_tani(lambda g, t=tam: t(g)[1])
    satirlar8.append({"deney_id": deney_id, "segmentasyon": ad, "parametre": param,
                      "MAE": kayit["MAE"], "RMSE": kayit["RMSE"], "Bias": kayit["Bias"], "Exact": kayit["Exact"],
                      **tani, "sure_ms": kayit["sure_ms"]})

tablo8 = pd.DataFrame(satirlar8).set_index("deney_id")
print("Geliştirme kümesi (kare başına ortalamalar). Referans ortalaması: 22.7 hücre\n")
print(tablo8.round(2).to_string())

In [ ]:
# Bölüm 8 (c): Segmentasyon çıktılarının görsel karşılaştırması
# Üst satır: tüm kare. Alt satır: Bölüm 7'deki en soluk hücrenin çevresi (aynı 300x300 bölge).
k = en_kotu
f, tra = dev_g15[k], dev_tra[k]
fig, ax = plt.subplots(2, len(SEGMENTASYONLAR), figsize=(3.4 * len(SEGMENTASYONLAR), 7.5))
for i, (deney_id, (ad, param, segment)) in enumerate(SEGMENTASYONLAR.items()):
    sayi, etiket = maske_say(segment(f))
    renkli = label2rgb(etiket, bg_label=0)
    renkli[find_boundaries(tra, mode="outer")] = [1, 0, 0]
    ax[0, i].imshow(renkli)
    ax[0, i].set_title(f"{deney_id} {ad}\n{param}\n{sayi} nesne (ref {dev_referans[k]})", fontsize=9)
    ax[1, i].imshow(renkli[kesit])
for a in ax.ravel():
    a.axis("off")
fig.suptitle(f"t{k:03d} - segmentasyon yöntemleri (Gaussian 15x15 sonrası) | renkli: bulunan nesneler, kırmızı: referans",
             fontsize=11)
plt.tight_layout()
plt.show()

## Bölüm 9 – Morfolojik İşlemler ve Nesne Ayırma

**Sabit tutulan önceki adımlar:** Orijinal yoğunluk → Gaussian 15×15 → Blok Otsu 8×8 (taban 4) = Bölüm 8'in en iyisi (E21). Yalnızca ikili maskeye uygulanan morfolojik işlem değişir; sonrasında 8-komşuluk bağlı bileşen + min. alan 50.

**Yapılandırma elemanı:** elips (hücreler yuvarlak). İki boyut: **5×5** (yarıçap 2) ve **11×11** (yarıçap 5). Hücre çapı ≈60 piksel olduğundan ikisi de hücreyi yok etmeyecek kadar küçük, gürültü lekelerini (< 100 piksel ≈ çap 10) etkileyecek kadar büyüktür.

| ID | İşlem | Beklenen etki |
|---|---|---|
| E23, E24 | Erosion 5×5, 11×11 | Nesneleri küçültür; ince köprüleri koparır, küçük lekeleri siler, soluk küçük hücreleri de silebilir |
| E25, E26 | Dilation 5×5, 11×11 | Nesneleri büyütür; delikleri kapatır ama yakın hücreleri birleştirir |
| E27, E28 | Opening 5×5, 11×11 | Erosion → dilation: küçük lekeleri siler, şekli korur |
| E29, E30 | Closing 5×5, 11×11 | Dilation → erosion: delikleri ve çatlakları kapatır, şekli korur |
| E31 | Opening → Closing (11×11) | İşlem sırasının etkisi |
| E32 | Closing → Opening (11×11) | İşlem sırasının etkisi |
| E33 | Delik doldurma | Hücre içi siyah noktaları (çekirdekçik) doldurur |
| E34 | Distance transform + watershed | Birbirine değen hücreleri ayırır; `min_distance` taranır |
| E35 | Closing 11×11 → delik doldurma → watershed | Watershed'den önce maskeyi düzeltmenin etkisi; `min_distance` taranır |

**Watershed nasıl çalışır?** Maskedeki her pikselin en yakın arka plana uzaklığı hesaplanır (distance transform). Hücre merkezleri bu haritada "tepe" olur. Her tepeye bir işaret konur ve bölgeler bu işaretlerden büyütülür; iki hücrenin buluştuğu "boyun" noktasında sınır oluşur. `min_distance`, iki tepe arasındaki en küçük uzaklıktır (≈ hücre yarıçapı).

In [ ]:
# Bölüm 9 (a): Morfolojik işlemler, işlem sırası, delik doldurma ve watershed
from skimage.segmentation import watershed
from skimage.feature import peak_local_max

dev_blok = {k: blok_otsu(f, 8, BLOK8_TABAN).astype(np.uint8) for k, f in dev_g15.items()}   # E21 maskeleri

def elips(boyut):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (boyut, boyut))

def morf(maske, islem, boyut):
    tip = {"erosion": cv2.MORPH_ERODE, "dilation": cv2.MORPH_DILATE,
           "opening": cv2.MORPH_OPEN, "closing": cv2.MORPH_CLOSE}[islem]
    return cv2.morphologyEx(maske, tip, elips(boyut))

def delik_doldur(maske):
    return ndimage.binary_fill_holes(maske).astype(np.uint8)

def watershed_ayir(maske, min_mesafe, min_alan=BASELINE_MIN_ALAN):
    """Distance transform + watershed. Çıktı: etiket görüntüsü (küçük parçalar atılmış)."""
    _, bilesenler = maske_say(maske, min_alan)                       # önce küçük lekeleri at
    uzaklik = ndimage.distance_transform_edt(bilesenler > 0)
    tepeler = peak_local_max(uzaklik, min_distance=min_mesafe, labels=bilesenler, exclude_border=False)
    isaretler = np.zeros(maske.shape, np.int32)
    isaretler[tuple(tepeler.T)] = np.arange(1, len(tepeler) + 1)
    etiket = watershed(-uzaklik, isaretler, mask=bilesenler > 0)
    kimlik, alan = np.unique(etiket[etiket > 0], return_counts=True)
    return np.where(np.isin(etiket, kimlik[alan >= min_alan]), etiket, 0)

def etiket_say(etiket):
    return len(np.unique(etiket)) - 1

# Watershed min_distance taraması (geliştirme kümesi)
def ws_hazirlik_yok(m):
    return m

def ws_hazirlik_kapama(m):
    return delik_doldur(morf(m, "closing", 11))

tarama_ws = []
for hazirlik_ad, hazirlik in [("ham maske", ws_hazirlik_yok), ("closing 11 + delik doldurma", ws_hazirlik_kapama)]:
    for md in [10, 15, 20, 25, 30, 35, 40, 50]:
        tahmin = [etiket_say(watershed_ayir(hazirlik(m), md)) for m in dev_blok.values()]
        tarama_ws.append({"hazırlık": hazirlik_ad, "min_distance": md,
                          **metrikler(tahmin, [dev_referans[k] for k in dev_blok])})
tarama_ws = pd.DataFrame(tarama_ws)
WS_MD_HAM = int(tarama_ws[tarama_ws["hazırlık"] == "ham maske"].set_index("min_distance").MAE.idxmin())
WS_MD_KAPAMA = int(tarama_ws[tarama_ws["hazırlık"] != "ham maske"].set_index("min_distance").MAE.idxmin())
print("Watershed min_distance taraması (geliştirme):")
print(tarama_ws.pivot(index="min_distance", columns="hazırlık", values="MAE").round(2).to_string())
print(f"Seçilen: ham maske -> {WS_MD_HAM}, closing+doldurma -> {WS_MD_KAPAMA}\n")

# Morfoloji deneyleri: her biri E21 maskesinden etiket görüntüsü üretir
MORFOLOJILER = {
    "E21": ("Yok",                       "-",            lambda m: maske_say(m)[1]),
    "E23": ("Erosion",                   "5x5",          lambda m: maske_say(morf(m, "erosion", 5))[1]),
    "E24": ("Erosion",                   "11x11",        lambda m: maske_say(morf(m, "erosion", 11))[1]),
    "E25": ("Dilation",                  "5x5",          lambda m: maske_say(morf(m, "dilation", 5))[1]),
    "E26": ("Dilation",                  "11x11",        lambda m: maske_say(morf(m, "dilation", 11))[1]),
    "E27": ("Opening",                   "5x5",          lambda m: maske_say(morf(m, "opening", 5))[1]),
    "E28": ("Opening",                   "11x11",        lambda m: maske_say(morf(m, "opening", 11))[1]),
    "E29": ("Closing",                   "5x5",          lambda m: maske_say(morf(m, "closing", 5))[1]),
    "E30": ("Closing",                   "11x11",        lambda m: maske_say(morf(m, "closing", 11))[1]),
    "E31": ("Opening -> Closing",        "11x11",        lambda m: maske_say(morf(morf(m, "opening", 11), "closing", 11))[1]),
    "E32": ("Closing -> Opening",        "11x11",        lambda m: maske_say(morf(morf(m, "closing", 11), "opening", 11))[1]),
    "E33": ("Delik doldurma",            "-",            lambda m: maske_say(delik_doldur(m))[1]),
    "E34": ("Watershed (ham maske)",     f"min_distance={WS_MD_HAM}",    lambda m: watershed_ayir(m, WS_MD_HAM)),
    "E35": ("Closing -> doldurma -> watershed", f"11x11, min_distance={WS_MD_KAPAMA}",
            lambda m: watershed_ayir(ws_hazirlik_kapama(m), WS_MD_KAPAMA)),
}

satirlar9 = []
for deney_id, (ad, param, islem) in MORFOLOJILER.items():
    tam = lambda g, i=islem: i(blok_otsu(on_isleme_g15(g), 8, BLOK8_TABAN).astype(np.uint8))
    if deney_id != "E21":
        deney_ekle(deney_id, "Orijinal", "Gaussian 15x15", f"Blok Otsu 8x8 (taban={BLOK8_TABAN})",
                   f"{ad} {param}", f"{param}; min_alan={BASELINE_MIN_ALAN}", lambda g, t=tam: etiket_say(t(g)))
    kayit = next(d for d in DENEYLER if d["deney_id"] == deney_id)
    tani = dev_tani(tam)
    satirlar9.append({"deney_id": deney_id, "morfoloji": f"{ad} {param}", "MAE": kayit["MAE"],
                      "RMSE": kayit["RMSE"], "Bias": kayit["Bias"], "Exact": kayit["Exact"],
                      **tani, "sure_ms": kayit["sure_ms"]})

tablo9 = pd.DataFrame(satirlar9).set_index("deney_id")
print("Geliştirme kümesi (kare başına ortalamalar). Referans ortalaması: 22.7 hücre\n")
print(tablo9.round(2).to_string())

In [ ]:
# Bölüm 9 (b): Görsel karşılaştırma - birbirine değen hücreler
# Bölge nesnel seçilir: E21 çıktısında birden fazla referans hücreyi içeren (birleşmiş) İLK nesne.
birlesik = None
for k, m in dev_blok.items():
    _, etiket = maske_say(m)
    for n in np.unique(etiket)[1:]:
        hucreler_n = np.unique(dev_tra[k][(etiket == n) & (dev_tra[k] > 0)])
        if len(hucreler_n) > 1:
            birlesik = (k, n, hucreler_n)
            break
    if birlesik:
        break

k, n, hucreler_n = birlesik
cy, cx = np.argwhere(maske_say(dev_blok[k])[1] == n).mean(axis=0).astype(int)
y0, x0 = np.clip([cy - 110, cx - 110], 0, 1024 - 220)
bolge = (slice(y0, y0 + 220), slice(x0, x0 + 220))
print(f"Seçilen bölge: t{k:03d}, birleşmiş nesne referans hücreleri {[int(h) for h in hucreler_n]} içeriyor")

fig, ax = plt.subplots(2, 7, figsize=(24, 8))
for a, (deney_id, (ad, param, islem)) in zip(ax.ravel(), MORFOLOJILER.items()):
    etiket = islem(dev_blok[k])
    renkli = label2rgb(etiket, bg_label=0)
    renkli[find_boundaries(dev_tra[k], mode="outer")] = [1, 0, 0]
    a.imshow(renkli[bolge])
    kayit = next(d for d in DENEYLER if d["deney_id"] == deney_id)
    a.set_title(f"{deney_id} {ad}\n{param}\nkare: {etiket_say(etiket)} nesne (ref {dev_referans[k]}) | MAE {kayit['MAE']:.1f}",
                fontsize=8.5)
    a.axis("off")
fig.suptitle(f"t{k:03d} - morfolojik işlemler (renkli: nesneler, kırmızı: referans hücreler)", fontsize=12)
plt.tight_layout()
plt.show()

# Watershed adımları (E35)
hazir = ws_hazirlik_kapama(dev_blok[k])
_, bilesenler = maske_say(hazir)
uzaklik = ndimage.distance_transform_edt(bilesenler > 0)
tepeler = peak_local_max(uzaklik, min_distance=WS_MD_KAPAMA, labels=bilesenler, exclude_border=False)
sonuc = watershed_ayir(hazir, WS_MD_KAPAMA)

fig, ax = plt.subplots(1, 4, figsize=(18, 4.8))
ax[0].imshow(dev_blok[k][bolge], cmap="gray"); ax[0].set_title("1) Blok Otsu maskesi (E21)")
ax[1].imshow(hazir[bolge], cmap="gray"); ax[1].set_title("2) Closing 11x11 + delik doldurma")
ax[2].imshow(uzaklik[bolge], cmap="magma")
icerde = [(y - y0, x - x0) for y, x in tepeler if y0 <= y < y0 + 220 and x0 <= x < x0 + 220]
if icerde:
    ax[2].plot([p[1] for p in icerde], [p[0] for p in icerde], "c+", ms=14, mew=2)
ax[2].set_title(f"3) Distance transform + tepeler (min_distance={WS_MD_KAPAMA})")
renkli = label2rgb(sonuc, bg_label=0)
renkli[find_boundaries(dev_tra[k], mode="outer")] = [1, 0, 0]
ax[3].imshow(renkli[bolge]); ax[3].set_title("4) Watershed sonucu + referans (kırmızı)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## Bölüm 10 – Kontrollü Deneyler ve Alternatif Pipeline'lar

Bölüm 6–9'daki deneyler (E01–E35) her aşamayı ayrı ayrı inceledi. Bu bölümde aşamalar **birleştirilerek tam pipeline** olarak denenir.

**Pipeline yapılandırması:** Her pipeline bir sözlükle (`cfg`) tanımlanır: yoğunluk → filtre → eşikleme → closing → delik doldurma → watershed → min. alan. Aynı sözlük Bölüm 12'de `final_pipeline.json` dosyasına yazılacaktır.

**Kontrollü deneyler** (her biri referans pipeline'dan **tek bir** değişiklik):

| ID | Değişiklik | Amaç |
|---|---|---|
| E36 | min. alan taraması (E35 üzerinde) | Watershed sonrası küçük parçaları ve gürültüyü atmak; gürültü lekeleri < 100 px, bölünme sonrası yavru hücreler 250–400 px |
| E37 | Yoğunluk: CLAHE | Yoğunluk dönüşümünün tam pipeline'a etkisi |
| E38 | Yoğunluk: contrast stretching | Aynı soru, global normalizasyon |
| E39 | Filtre: Gaussian 9×9 | Filtre boyutunun tam pipeline'a etkisi |
| E40 | Filtre: Median 9×9 | Filtre türünün etkisi |
| E41 | Eşik: Li | Parametresiz global eşik |
| E42 | Eşik: sabit T=7 | En basit eşik |
| E43 | Eşik: interpolasyonlu Blok Otsu | 10 (b)'de görülen blok sınırı yapaylığını gidermek (10 (c)) |

Referans pipeline = **E36** (E35 + seçilen min. alan). E37–E43 bu referanstan yalnızca belirtilen adımı değiştirir.

**Aday pipeline'lar** (Bölüm 11'de doğrulama kümesinde karşılaştırılacak):
- **Pipeline A** = E36: Gaussian 15×15 → Blok Otsu 8×8 → closing → doldurma → watershed
- **Pipeline B** = E40: Median 9×9 → Blok Otsu 8×8 → closing → doldurma → watershed
- **Pipeline C** = E41: Gaussian 15×15 → Li → closing → doldurma → watershed
- **Pipeline D** = E42: Gaussian 15×15 → sabit eşik T=7 → closing → doldurma → watershed

Doküman gereği tüm geliştirme görüntüleri görsel olarak incelenir (10 (b)). Bütün deneyler en sonda `experiments.csv` dosyasına yazılır (10 (c)).


In [ ]:
# Bölüm 10 (a): Yapılandırılabilir tam pipeline ve kontrollü deneyler (E36-E42)
import json

def pipeline_etiket(g, cfg):
    """Ham görüntü + yapılandırma -> etiket görüntüsü (her nesne = bir hücre)."""
    yogunluk = {"orijinal": lambda x: x, "clahe": CLAHE.apply, "contrast_stretch": contrast_stretch}
    f = yogunluk[cfg["yogunluk"]](g)
    if cfg["filtre"] == "gaussian":
        f = cv2.GaussianBlur(f, (cfg["filtre_boyut"], cfg["filtre_boyut"]), 0)
    else:
        f = cv2.medianBlur(f, cfg["filtre_boyut"])
    if cfg["esik"] == "blok_otsu":
        maske = blok_otsu(f, cfg["n_blok"], cfg["taban"])
    elif cfg["esik"] == "blok_otsu_interp":
        maske = blok_otsu_interp(f, cfg["n_blok"], cfg["taban"])
    elif cfg["esik"] == "li":
        maske = li_esik(f)
    else:
        maske = sabit_esik(f, cfg["T"])
    maske = morf(maske.astype(np.uint8), "closing", cfg["kapama"])
    maske = delik_doldur(maske)
    return watershed_ayir(maske, cfg["min_distance"], cfg["min_alan"])

def blok_otsu_interp(f, n_blok, taban):
    """Blok Otsu'nun yumuşatılmış hali: blok eşikleri (taban ile) doğrusal interpolasyonla
    her piksele dağıtılır; blok sınırındaki ani eşik sıçraması ortadan kalkar (E43)."""
    h, w = f.shape[0] // n_blok, f.shape[1] // n_blok
    esikler = np.zeros((n_blok, n_blok), np.float32)
    for r in range(n_blok):
        for c in range(n_blok):
            parca = f[r * h:(r + 1) * h, c * w:(c + 1) * w]
            esikler[r, c] = max(cv2.threshold(parca, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0], taban)
    esik_haritasi = cv2.resize(esikler, (f.shape[1], f.shape[0]), interpolation=cv2.INTER_LINEAR)
    return f > esik_haritasi

def pipeline_say(g, cfg):
    return etiket_say(pipeline_etiket(g, cfg))

def cfg_ekle(deney_id, cfg):
    """Yapılandırmayı deney kaydına çevirir (tablo sütunları otomatik doldurulur)."""
    esik_ad = {"blok_otsu": f"Blok Otsu {cfg['n_blok']}x{cfg['n_blok']} (taban={cfg['taban']})",
               "blok_otsu_interp": f"Blok Otsu {cfg['n_blok']}x{cfg['n_blok']} interpolasyonlu (taban={cfg['taban']})",
               "li": "Li", "sabit": f"Sabit eşik T={cfg.get('T')}"}[cfg["esik"]]
    yog_ad = {"orijinal": "Orijinal", "clahe": "CLAHE", "contrast_stretch": "Contrast stretching"}[cfg["yogunluk"]]
    return deney_ekle(deney_id, yog_ad, f"{cfg['filtre'].capitalize()} {cfg['filtre_boyut']}x{cfg['filtre_boyut']}",
                      esik_ad, f"Closing {cfg['kapama']}x{cfg['kapama']} -> doldurma -> watershed",
                      f"min_distance={cfg['min_distance']}; min_alan={cfg['min_alan']}",
                      lambda g: pipeline_say(g, cfg))

E35_CFG = {"yogunluk": "orijinal", "filtre": "gaussian", "filtre_boyut": 15,
           "esik": "blok_otsu", "n_blok": 8, "taban": BLOK8_TABAN,
           "kapama": 11, "min_distance": WS_MD_KAPAMA, "min_alan": BASELINE_MIN_ALAN}

# E36: min. alan taraması (yalnızca geliştirme kümesi)
tarama_alan = pd.DataFrame([{"min_alan": a, **metrikler([pipeline_say(g, {**E35_CFG, "min_alan": a})
                                                         for g in dev_goruntuler.values()],
                                                        [dev_referans[k] for k in dev_goruntuler])}
                            for a in [50, 75, 100, 150, 200, 300, 400, 600]])
SON_MIN_ALAN = int(tarama_alan.set_index("min_alan").MAE.idxmin())   # eşitlikte en küçük değer
print("min. alan taraması (E35 üzerinde):")
print(tarama_alan.round(2).to_string(index=False))
print(f"Seçilen min. alan: {SON_MIN_ALAN}\n")

REFERANS_CFG = {**E35_CFG, "min_alan": SON_MIN_ALAN}
DENEYLER10 = {
    "E36": REFERANS_CFG,
    "E37": {**REFERANS_CFG, "yogunluk": "clahe"},
    "E38": {**REFERANS_CFG, "yogunluk": "contrast_stretch"},
    "E39": {**REFERANS_CFG, "filtre_boyut": 9},
    "E40": {**REFERANS_CFG, "filtre": "median", "filtre_boyut": 9},
    "E41": {**REFERANS_CFG, "esik": "li"},
    "E42": {**REFERANS_CFG, "esik": "sabit", "T": EN_IYI_T},
}
ADAYLAR = {"Pipeline A": "E36", "Pipeline B": "E40", "Pipeline C": "E41", "Pipeline D": "E42"}

satirlar10 = []
for deney_id, cfg in DENEYLER10.items():
    kayit = cfg_ekle(deney_id, cfg)
    degisiklik = {k: v for k, v in cfg.items() if REFERANS_CFG.get(k) != v} if deney_id != "E36" else {"min_alan": SON_MIN_ALAN}
    satirlar10.append({"deney_id": deney_id, "değişiklik": str(degisiklik), "MAE": kayit["MAE"], "RMSE": kayit["RMSE"],
                       "Bias": kayit["Bias"], **dev_tani(lambda g, c=cfg: pipeline_etiket(g, c)),
                       "sure_ms": kayit["sure_ms"]})
tablo10 = pd.DataFrame(satirlar10).set_index("deney_id")
print("Kontrollü deneyler (referans = E36):")
print(tablo10.round(2).to_string())


In [ ]:
# Bölüm 10 (b): Tüm geliştirme görüntülerinin görsel incelemesi (Pipeline A)
from skimage.segmentation import find_boundaries

def sinir(etiket, kalinlik=3):
    return cv2.dilate(find_boundaries(etiket, mode="inner").astype(np.uint8), elips(kalinlik)) > 0

cfg_a = DENEYLER10[ADAYLAR["Pipeline A"]]
fig, ax = plt.subplots(2, 5, figsize=(25, 12.5), constrained_layout=True)
for a, (k, g) in zip(ax.ravel(), dev_goruntuler.items()):
    etiket = pipeline_etiket(g, cfg_a)
    tani = tani_hesapla(etiket, dev_tra[k])
    gorsel = np.dstack([gosterim_icin(g)] * 3)                       # gosterim_icin 0-1 aralığında döner
    gorsel[sinir(etiket)] = [0, 1, 0]                                 # yeşil: bulunan nesnelerin sınırı
    gorsel[find_boundaries(dev_tra[k], mode="outer")] = [1, 0, 0]    # kırmızı: referans işaretlerinin sınırı
    a.imshow(gorsel)
    a.set_title(f"t{k:03d}: tahmin {etiket_say(etiket)} / ref {dev_referans[k]}\n"
                f"kaçırılan {tani['kacirilan']}, bölünme {tani['bolunme']}, "
                f"birleşme {tani['birlesme']}, fazla {tani['fazla']}", fontsize=11)
    a.axis("off")
fig.suptitle("Pipeline A - geliştirme kümesinin tamamı (yeşil: nesne sınırı, kırmızı: referans işareti)", fontsize=14)
plt.show()

# Dört aday pipeline - aynı kare (adayların toplam hatasının en büyük olduğu geliştirme karesi)
farklar = {k: sum(abs(pipeline_say(g, DENEYLER10[d]) - dev_referans[k]) for d in ADAYLAR.values())
           for k, g in dev_goruntuler.items()}
zor = max(farklar, key=farklar.get)
fig, ax = plt.subplots(1, 4, figsize=(22, 6.5), constrained_layout=True)
for a, (ad, d) in zip(ax, ADAYLAR.items()):
    etiket = pipeline_etiket(dev_goruntuler[zor], DENEYLER10[d])
    renkli = label2rgb(etiket, bg_label=0)
    renkli[dev_tra[zor] > 0] = [1, 1, 1]
    a.imshow(renkli)
    a.set_title(f"{ad} ({d})\ntahmin {etiket_say(etiket)} / ref {dev_referans[zor]}")
    a.axis("off")
fig.suptitle(f"t{zor:03d} - aday pipeline'lar (beyaz: referans işaretleri)", fontsize=13)
plt.show()


In [ ]:
# Bölüm 10 (c): Blok sınırı yapaylığı ve interpolasyonlu Blok Otsu (E43); experiments.csv
# Sorun: Blok Otsu her 128x128 blokta ayrı eşik kullanır. Blok sınırına denk gelen soluk bir hücrenin
# iki yarısı farklı eşiklerle kesilir; yarısı kaybolur ve düz bir kesik çizgi oluşur.
# Denenen çözüm: blok eşiklerini doğrusal interpolasyonla yumuşatmak (CLAHE'deki yöntem).

E43_CFG = {**REFERANS_CFG, "esik": "blok_otsu_interp"}
DENEYLER10["E43"] = E43_CFG

tarama_interp = []
for taban in [3, 4, 5, 6]:
    cfg = {**E43_CFG, "taban": taban}
    tahmin = [pipeline_say(g, cfg) for g in dev_goruntuler.values()]
    tarama_interp.append({"taban": taban, **metrikler(tahmin, [dev_referans[k] for k in dev_goruntuler]),
                          **dev_tani(lambda g, c=cfg: pipeline_etiket(g, c))})
print("İnterpolasyonlu Blok Otsu - taban taraması (karşılaştırma: E36 klasik Blok Otsu, taban 4):")
print(pd.DataFrame(tarama_interp).round(2).to_string(index=False))
print(tablo10.loc[["E36"], ["MAE", "RMSE", "Bias", "bulunan", "kacirilan", "bolunme", "birlesme", "fazla"]].round(2).to_string())

kayit = cfg_ekle("E43", E43_CFG)   # kontrollü deney: E36'dan tek fark interpolasyon (taban 4 aynı)
print(f"\nE43 (taban {E43_CFG['taban']}): MAE {kayit['MAE']:.1f}, RMSE {kayit['RMSE']:.2f}, süre {kayit['sure_ms']:.0f} ms")

# Eşik haritası: klasik (basamaklı) ve interpolasyonlu (yumuşak) - en çok hücre kaçırılan kare
k = max(dev_goruntuler, key=lambda k: tani_hesapla(pipeline_etiket(dev_goruntuler[k], REFERANS_CFG), dev_tra[k])["kacirilan"])
f = on_isleme_g15(dev_goruntuler[k])
h = f.shape[0] // 8
esik_klasik = np.kron(np.array([[max(cv2.threshold(f[r*h:(r+1)*h, c*h:(c+1)*h], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0],
                                     BLOK8_TABAN) for c in range(8)] for r in range(8)]), np.ones((h, h)))
esik_interp = cv2.resize(esik_klasik[h // 2::h, h // 2::h].astype(np.float32), f.shape[::-1], interpolation=cv2.INTER_LINEAR)
fig, ax = plt.subplots(1, 4, figsize=(24, 6.2), constrained_layout=True)
for a, (baslik, harita) in zip(ax[:2], [("Klasik: blok başına tek eşik", esik_klasik),
                                         ("İnterpolasyonlu: yumuşak eşik haritası", esik_interp)]):
    im = a.imshow(harita, cmap="viridis", vmin=0, vmax=esik_klasik.max())
    a.set_title(f"t{k:03d} - {baslik}")
    fig.colorbar(im, ax=a, fraction=0.046)
for a, (ad, cfg) in zip(ax[2:], [("E36 klasik", REFERANS_CFG), ("E43 interpolasyonlu", E43_CFG)]):
    etiket = pipeline_etiket(dev_goruntuler[k], cfg)
    gorsel = np.dstack([gosterim_icin(dev_goruntuler[k])] * 3)
    gorsel[sinir(etiket)] = [0, 1, 0]
    gorsel[find_boundaries(dev_tra[k], mode="outer")] = [1, 0, 0]
    tani = tani_hesapla(etiket, dev_tra[k])
    a.imshow(gorsel)
    for i in range(1, 8):
        a.axhline(i * h, color="yellow", lw=0.5, alpha=0.6); a.axvline(i * h, color="yellow", lw=0.5, alpha=0.6)
    a.set_title(f"{ad}: tahmin {etiket_say(etiket)} / ref {dev_referans[k]}, kaçırılan {tani['kacirilan']}")
for a in ax:
    a.axis("off")
plt.show()

# experiments.csv: tüm deneyler (E01-E43)
deney_tablosu = pd.DataFrame(DENEYLER).sort_values("deney_id")
deney_tablosu["dev_tahminler"] = deney_tablosu.tahminler.apply(
    lambda t: ";".join(f"t{k:03d}={int(v)}" for k, v in zip(dev_goruntuler, t)))
deney_tablosu = deney_tablosu.drop(columns="tahminler").round(3)
deney_tablosu.to_csv("experiments.csv", index=False)
print(f"\nexperiments.csv yazıldı: {len(deney_tablosu)} deney (E01-{deney_tablosu.deney_id.iloc[-1]})")

## Bölüm 11 – Doğrulama ve Nihai Pipeline Seçimi

Bölüm 10'daki 4 aday pipeline, **ilk kez** 5 doğrulama karesinde çalıştırılır. Bu karelerde hiçbir parametre seçilmedi; geliştirme kümesinde seçilen ayarlar **değiştirilmeden** kullanılır.

| Aday | Deney | Filtre | Eşik | Sonrası (hepsinde aynı) |
|---|---|---|---|---|
| Pipeline A | E36 | Gaussian 15×15 | Blok Otsu 8×8 (taban 4) | closing 11×11 → delik doldurma → watershed (md 25) → min. alan 100 |
| Pipeline B | E40 | Median 9×9 | Blok Otsu 8×8 (taban 4) | aynı |
| Pipeline C | E41 | Gaussian 15×15 | Li | aynı |
| Pipeline D | E42 | Gaussian 15×15 | Sabit eşik T=7 | aynı |

**Zorunlu metrikler:** MAE, RMSE ve kare başına ortalama işlem süresi. Ek olarak Bias, Exact ve tanı ölçütleri (kaçırılan, bölünme, birleşme, fazla) verilir.

**Seçim kuralı (doküman):** En düşük doğrulama MAE'si. Eşitlik olursa sırasıyla RMSE, işlem süresi ve pipeline karmaşıklığı.

Baseline (Otsu + bağlı bileşen, min. alan 50) karşılaştırma için tabloya eklenir, ancak aday değildir.

Seçimden sonra nihai pipeline ve baseline **sabitlenir**; Bölüm 12'de `final_pipeline.json` dosyasına yazılır.

In [ ]:
# Bölüm 11 (a): Aday pipeline'ların doğrulama kümesinde karşılaştırılması -> validation_results.csv
dogrulama_kareleri = secim[secim.group == "validation"].reset_index(drop=True)
val_goruntuler = {s.frame: goruntu_oku(VERI / s.image_file) for s in dogrulama_kareleri.itertuples()}
val_tra = {s.frame: referans_oku(s) for s in dogrulama_kareleri.itertuples()}
val_referans = referanslar.set_index("frame").reference_count
val_gercek = [int(val_referans[k]) for k in val_goruntuler]
print("Doğrulama kareleri:", ", ".join(f"t{k:03d} (ref {val_referans[k]})" for k in val_goruntuler), "\n")

def dogrulama_degerlendir(etiket_fonksiyonu):
    """Doğrulama karelerinde tahmin, süre ve tanı ölçütleri."""
    tahminler, sureler, toplam_tani = [], [], Counter()
    for k, g in val_goruntuler.items():
        t0 = time.perf_counter()
        etiket = etiket_fonksiyonu(g)
        sureler.append(time.perf_counter() - t0)
        tahminler.append(etiket_say(etiket))
        toplam_tani.update(tani_hesapla(etiket, val_tra[k]))
    return tahminler, 1000 * np.mean(sureler), {a: b / len(val_goruntuler) for a, b in toplam_tani.items()}

satirlar11 = []
adaylar_ve_baseline = {**{ad: DENEYLER10[d] for ad, d in ADAYLAR.items()}, "Baseline": None}
for ad, cfg in adaylar_ve_baseline.items():
    if cfg is None:
        fonk = lambda g: baseline_segment(g, BASELINE_MIN_ALAN)[2]
        deney_id = "E01"
    else:
        fonk = lambda g, c=cfg: pipeline_etiket(g, c)
        deney_id = ADAYLAR[ad]
    tahminler, sure_ms, tani = dogrulama_degerlendir(fonk)
    dev_kayit = next(d for d in DENEYLER if d["deney_id"] == deney_id)
    satirlar11.append({"pipeline": ad, "deney_id": deney_id, **metrikler(tahminler, val_gercek),
                       "ort_sure_ms": sure_ms, **tani, "dev_MAE": dev_kayit["MAE"],
                       "val_tahminler": ";".join(f"t{k:03d}={v}" for k, v in zip(val_goruntuler, tahminler))})

dogrulama = pd.DataFrame(satirlar11)

# Seçim kuralı: en düşük MAE; eşitlikte RMSE, sonra süre (yalnızca adaylar arasında)
adaylar_df = dogrulama[dogrulama.pipeline != "Baseline"].copy()
adaylar_df["_mae"] = adaylar_df.MAE.round(3)
adaylar_df["_rmse"] = adaylar_df.RMSE.round(3)
NIHAI_AD = adaylar_df.sort_values(["_mae", "_rmse", "ort_sure_ms"]).pipeline.iloc[0]
dogrulama["secim"] = np.where(dogrulama.pipeline == NIHAI_AD, "SEÇİLDİ", "")

pd.set_option("display.width", 250)
print(dogrulama.drop(columns="val_tahminler").set_index("pipeline").round(2).to_string())
print("\nKare bazında tahminler (doğrulama):")
print(pd.DataFrame({"ref": val_gercek, **{r.pipeline: [int(x.split("=")[1]) for x in r.val_tahminler.split(";")]
                                          for r in dogrulama.itertuples()}},
                   index=[f"t{k:03d}" for k in val_goruntuler]).to_string())

dogrulama.round(3).to_csv("validation_results.csv", index=False)
print("\nvalidation_results.csv yazıldı.")

# Nihai pipeline ve baseline sabitlenir (Bölüm 12'de final_pipeline.json'a yazılacak)
NIHAI_DENEY = ADAYLAR[NIHAI_AD]
ESIK_PARAMETRELERI = {"blok_otsu": ["n_blok", "taban"], "blok_otsu_interp": ["n_blok", "taban"], "li": [], "sabit": ["T"]}
NIHAI_CFG = {a: v for a, v in DENEYLER10[NIHAI_DENEY].items()          # yalnızca kullanılan parametreler
             if a not in ["n_blok", "taban", "T"] or a in ESIK_PARAMETRELERI[DENEYLER10[NIHAI_DENEY]["esik"]]}
BASELINE_CFG = {"yontem": "Global Otsu + 8-komşuluk bağlı bileşen", "min_alan": BASELINE_MIN_ALAN}
print(f"\nNİHAİ PIPELINE: {NIHAI_AD} ({NIHAI_DENEY})")
print(json.dumps(NIHAI_CFG, ensure_ascii=False, indent=2))

In [ ]:
# Bölüm 11 (b): Doğrulama sonuçlarının görselleştirilmesi
fig, ax = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
x = np.arange(len(val_goruntuler))
ax[0].plot(x, val_gercek, "k*-", ms=14, lw=2, label="Referans")
isaretler = {"Pipeline A": "o", "Pipeline B": "s", "Pipeline C": "^", "Pipeline D": "D", "Baseline": "x"}
for r in dogrulama.itertuples():
    tahmin = [int(v.split("=")[1]) for v in r.val_tahminler.split(";")]
    ax[0].plot(x, tahmin, marker=isaretler[r.pipeline], ls="--", alpha=0.8,
               lw=2.5 if r.pipeline == NIHAI_AD else 1, label=f"{r.pipeline} (MAE {r.MAE:.1f})")
ax[0].set_xticks(x, [f"t{k:03d}" for k in val_goruntuler])
ax[0].set_ylabel("Hücre sayısı"); ax[0].set_title("Doğrulama kareleri: referans ve tahminler")
ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)

adaylar_sirali = dogrulama[dogrulama.pipeline != "Baseline"]
renkler = ["tab:green" if p == NIHAI_AD else "tab:gray" for p in adaylar_sirali.pipeline]
ax[1].bar(np.arange(len(adaylar_sirali)) - 0.2, adaylar_sirali.dev_MAE, 0.4, color="lightsteelblue", label="Geliştirme MAE")
ax[1].bar(np.arange(len(adaylar_sirali)) + 0.2, adaylar_sirali.MAE, 0.4, color=renkler, label="Doğrulama MAE")
for i, r in enumerate(adaylar_sirali.itertuples()):
    ax[1].text(i + 0.2, r.MAE + 0.05, f"{r.MAE:.1f}\n{r.ort_sure_ms:.0f} ms", ha="center", fontsize=9)
ax[1].set_xticks(np.arange(len(adaylar_sirali)), adaylar_sirali.pipeline)
ax[1].set_ylabel("MAE"); ax[1].set_title("Geliştirme ve doğrulama MAE (yeşil: seçilen)")
ax[1].legend(); ax[1].grid(alpha=0.3, axis="y")
plt.show()

# Seçilen pipeline'ın 5 doğrulama karesindeki çıktısı
fig, ax = plt.subplots(1, 5, figsize=(25, 6), constrained_layout=True)
for a, (k, g) in zip(ax, val_goruntuler.items()):
    etiket = pipeline_etiket(g, NIHAI_CFG)
    tani = tani_hesapla(etiket, val_tra[k])
    gorsel = np.dstack([gosterim_icin(g)] * 3)
    gorsel[sinir(etiket)] = [0, 1, 0]
    gorsel[find_boundaries(val_tra[k], mode="outer")] = [1, 0, 0]
    a.imshow(gorsel)
    a.set_title(f"t{k:03d}: tahmin {etiket_say(etiket)} / ref {val_referans[k]}\n"
                f"kaçırılan {tani['kacirilan']}, bölünme {tani['bolunme']}, "
                f"birleşme {tani['birlesme']}, fazla {tani['fazla']}", fontsize=10)
    a.axis("off")
fig.suptitle(f"{NIHAI_AD} - doğrulama kareleri (yeşil: nesne sınırı, kırmızı: referans işareti)", fontsize=13)
plt.show()

## Bölüm 12 – Nihai Pipeline'ın Kilitlenmesi ve Test Tahminlerinin Kaydedilmesi

Bu bölüm test verisinin sızmasını önlemek için zorunludur. Sırası:

1. **12 (a)** – Nihai pipeline aşama aşama gösterilir (rapor Şekil 4, bir geliştirme karesi üzerinde).
2. **12 (b)** – Nihai pipeline ve baseline parametreleri `final_pipeline.json` dosyasına yazılır.
3. **12 (b)** – Sabitlenmiş pipeline ve baseline, 15 test karesinin (Seq 02) tamamında çalıştırılır. **Test referansları okunmaz** (`TEST_REFERANSLARI_KILITLI = True` hâlâ geçerlidir). Tahminler `test_predictions_locked.csv` dosyasına yazılır.
4. Bu dosyalar **ayrı bir commit** ile GitHub'a yüklenir. Commit zamanı, yöntemin test referanslarına bakılmadan belirlendiğinin kanıtıdır.

**Kilit koruması:** Dosyalar bir kez oluşturulduktan sonra notebook yeniden çalıştırılırsa **üzerlerine yazılmaz**. Bunun yerine tahminler yeniden hesaplanır ve kilitli dosyadaki değerlerle birebir aynı olduğu doğrulanır (tekrarlanabilirlik kontrolü).

Test görüntüleri bu bölümde **gösterilmez**; görsel inceleme Bölüm 13'te kilit açıldıktan sonra yapılır.

In [ ]:
# Bölüm 12 (a): Nihai pipeline'ın aşama aşama çıktıları (rapor Şekil 4) - geliştirme karesi
ornek_kare = 41
g = dev_goruntuler[ornek_kare]

adim_filtre = cv2.GaussianBlur(g, (NIHAI_CFG["filtre_boyut"], NIHAI_CFG["filtre_boyut"]), 0)
adim_esik = sabit_esik(adim_filtre, NIHAI_CFG["T"]).astype(np.uint8)
adim_kapama = morf(adim_esik, "closing", NIHAI_CFG["kapama"])
adim_doldur = delik_doldur(adim_kapama)
_, adim_bilesen = maske_say(adim_doldur, NIHAI_CFG["min_alan"])
adim_uzaklik = ndimage.distance_transform_edt(adim_bilesen > 0)
adim_tepeler = peak_local_max(adim_uzaklik, min_distance=NIHAI_CFG["min_distance"], labels=adim_bilesen, exclude_border=False)
adim_etiket = pipeline_etiket(g, NIHAI_CFG)

fig, ax = plt.subplots(2, 4, figsize=(24, 12.5), constrained_layout=True)
ax = ax.ravel()
ax[0].imshow(g, cmap="gray", vmin=0, vmax=255); ax[0].set_title("1) Ham görüntü (0-255)")
ax[1].imshow(gosterim_icin(g), cmap="gray"); ax[1].set_title("   Ham görüntü (yalnızca gösterim için parlatılmış)")
ax[2].imshow(adim_filtre, cmap="gray", vmin=0, vmax=30); ax[2].set_title(f"2) Gaussian {NIHAI_CFG['filtre_boyut']}x{NIHAI_CFG['filtre_boyut']} (gösterim 0-30)")
ax[3].imshow(adim_esik, cmap="gray"); ax[3].set_title(f"3) Sabit eşik T={NIHAI_CFG['T']}")
ax[4].imshow(adim_doldur, cmap="gray"); ax[4].set_title(f"4) Closing {NIHAI_CFG['kapama']}x{NIHAI_CFG['kapama']} + delik doldurma")
ax[5].imshow(adim_uzaklik, cmap="magma"); ax[5].plot(adim_tepeler[:, 1], adim_tepeler[:, 0], "c+", ms=10, mew=2)
ax[5].set_title(f"5) Distance transform + tepeler (min_distance={NIHAI_CFG['min_distance']})")
ax[6].imshow(label2rgb(adim_etiket, bg_label=0)); ax[6].set_title(f"6) Watershed + min. alan {NIHAI_CFG['min_alan']}")
gorsel = np.dstack([gosterim_icin(g)] * 3)
gorsel[sinir(adim_etiket)] = [0, 1, 0]
gorsel[find_boundaries(dev_tra[ornek_kare], mode="outer")] = [1, 0, 0]
ax[7].imshow(gorsel); ax[7].set_title(f"7) Sonuç: {etiket_say(adim_etiket)} hücre (referans {dev_referans[ornek_kare]})")
for a in ax:
    a.axis("off")
fig.suptitle(f"Nihai pipeline ({NIHAI_AD}) - aşama aşama çıktılar, t{ornek_kare:03d} (geliştirme karesi)", fontsize=14)
plt.show()

In [ ]:
# Bölüm 12 (b): final_pipeline.json + test tahminleri (referanslar OKUNMADAN) -> test_predictions_locked.csv
import hashlib, platform

assert TEST_REFERANSLARI_KILITLI, "Bu hücre yalnızca test referansları kilitliyken çalışmalıdır."
JSON_YOLU, KILITLI_YOLU = Path("final_pipeline.json"), Path("test_predictions_locked.csv")

dev_nihai = next(d for d in DENEYLER if d["deney_id"] == NIHAI_DENEY)
val_nihai = dogrulama.set_index("pipeline").loc[NIHAI_AD]
final_pipeline = {
    "nihai_pipeline": NIHAI_AD,
    "deney_id": NIHAI_DENEY,
    "adim_sirasi": ["yogunluk", "filtre", "esik", "closing", "delik_doldurma", "distance_transform_watershed",
                    "baglanti_8_komsuluk", "min_alan_filtresi", "sayim"],
    "parametreler": {a: (int(v) if isinstance(v, (int, np.integer)) else v) for a, v in NIHAI_CFG.items()},
    "yapilandirma_elemani": "elips",
    "baseline": {"yontem": "Global Otsu + 8-komşuluk bağlı bileşen", "min_alan": int(BASELINE_MIN_ALAN)},
    "secim_kurali": "En düşük doğrulama MAE; eşitlikte RMSE, işlem süresi, karmaşıklık",
    "gelistirme": {"MAE": round(float(dev_nihai["MAE"]), 3), "RMSE": round(float(dev_nihai["RMSE"]), 3)},
    "dogrulama": {"MAE": round(float(val_nihai.MAE), 3), "RMSE": round(float(val_nihai.RMSE), 3),
                  "ort_sure_ms": round(float(val_nihai.ort_sure_ms), 1)},
    "seed": int(SEED),
    "ortam": {"python": platform.python_version(), "numpy": np.__version__, "opencv": cv2.__version__,
              "scipy": __import__("scipy").__version__, "scikit-image": __import__("skimage").__version__},
}

# ---- Test tahminleri: yalnızca görüntüler okunur, referans dosyalarına dokunulmaz ----
test_kareleri = secim[secim.group == "test"].reset_index(drop=True)
satirlar12 = []
for s in test_kareleri.itertuples():
    g = goruntu_oku(VERI / s.image_file)
    t0 = time.perf_counter(); baseline_tahmin = baseline_segment(g, BASELINE_MIN_ALAN)[0]; t1 = time.perf_counter()
    nihai_tahmin = etiket_say(pipeline_etiket(g, NIHAI_CFG)); t2 = time.perf_counter()
    satirlar12.append({"group": s.group, "sequence": s.sequence, "frame": int(s.frame), "test_block": int(s.test_block),
                       "image_file": s.image_file, "baseline_tahmin": int(baseline_tahmin), "nihai_tahmin": int(nihai_tahmin),
                       "baseline_sure_ms": round(1000 * (t1 - t0), 2), "nihai_sure_ms": round(1000 * (t2 - t1), 2)})
test_tahmin = pd.DataFrame(satirlar12)

def sha256(yol):
    return hashlib.sha256(yol.read_bytes()).hexdigest()

if not KILITLI_YOLU.exists():
    # İlk çalıştırma: dosyalar oluşturulur (kilitleme)
    JSON_YOLU.write_text(json.dumps(final_pipeline, ensure_ascii=False, indent=2), encoding="utf-8")
    test_tahmin.to_csv(KILITLI_YOLU, index=False)
    print("KİLİTLENDİ: final_pipeline.json ve test_predictions_locked.csv oluşturuldu.")
else:
    # Sonraki çalıştırmalar: üzerine yazılmaz, yeniden hesaplanan tahminler kilitli dosyayla karşılaştırılır
    kilitli = pd.read_csv(KILITLI_YOLU)
    ayni = (kilitli[["frame", "baseline_tahmin", "nihai_tahmin"]].values ==
            test_tahmin[["frame", "baseline_tahmin", "nihai_tahmin"]].values).all()
    json_ayni = json.loads(JSON_YOLU.read_text(encoding="utf-8"))["parametreler"] == final_pipeline["parametreler"]
    assert ayni and json_ayni, "UYARI: Yeniden hesaplanan sonuçlar kilitli dosyalarla uyuşmuyor!"
    print("Kilitli dosyalar zaten var; üzerine yazılmadı. Yeniden hesaplanan tahminler ve parametreler kilitli dosyalarla AYNI.")

print(f"  final_pipeline.json          SHA-256: {sha256(JSON_YOLU)[:16]}...")
print(f"  test_predictions_locked.csv  SHA-256: {sha256(KILITLI_YOLU)[:16]}...\n")
print(json.dumps(final_pipeline, ensure_ascii=False, indent=2))
print("\nTest tahminleri (referanslar okunmadı):")
print(pd.read_csv(KILITLI_YOLU).drop(columns=["group", "image_file"]).to_string(index=False))

## Bölüm 13 – Bağımsız Test Performansının Ölçülmesi

**Kilit açma koşulları** (13 (a) bunları otomatik kontrol eder, biri sağlanmazsa durur):
1. `final_pipeline.json` ve `test_predictions_locked.csv` mevcut.
2. İkisi de git'te commit'lenmiş ve commit'ten sonra **değiştirilmemiş**.
3. Commit GitHub'a gönderilmiş (yerel dal uzak daldan ileride değil).

Koşullar sağlanınca `TEST_REFERANSLARI_KILITLI = False` yapılır ve 15 test karesinin TRA referansları **ilk kez** okunur.

**Önemli:** Değerlendirmede tahminler **yeniden hesaplanmaz**; doğrudan kilitli dosyadan okunur. Süreler de kilitli dosyadakilerdir.

**Metrikler:** MAE, RMSE, Bias (ortalama işaretli hata), Exact-count Accuracy, görüntü başına ortalama süre.

**Çıktılar:**
- `test_results.csv` (kare bazında: gerçek sayı, tahminler, mutlak hatalar)
- `test_summary.csv` (özet metrikler)
- 15 test karesinin tamamının görsel incelemesi (13 (b))

Bu aşamadan sonra algoritma, parametreler ve tahminler **değiştirilmez**.

In [ ]:
# Bölüm 13 (a): Kilit kontrolü, test referanslarının açılması ve test metrikleri
import subprocess

def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, encoding="utf-8").stdout.strip()

# ---- 1) Kilit koşulları ----
assert JSON_YOLU.exists() and KILITLI_YOLU.exists(), "Kilitli dosyalar yok! Önce Bölüm 12 çalıştırılmalı."
izlenen = git("ls-files", str(JSON_YOLU), str(KILITLI_YOLU)).split()
assert len(izlenen) == 2, "Kilitli dosyalar git'te commit'lenmemiş!"
assert git("status", "--porcelain", str(JSON_YOLU), str(KILITLI_YOLU)) == "", "Kilitli dosyalar commit'ten sonra değiştirilmiş!"
assert "ahead" not in git("status", "-sb").splitlines()[0], "Kilit commit'i GitHub'a gönderilmemiş (git push)!"
kilit_commit = git("log", "-1", "--format=%h | %ci | %s", "--", str(KILITLI_YOLU))
print("Kilit koşulları sağlandı.")
print(f"  Kilit commit'i: {kilit_commit}\n")

# ---- 2) Kilidi aç ve test referanslarını oku ----
TEST_REFERANSLARI_KILITLI = False
test_kareleri = secim[secim.group == "test"].reset_index(drop=True)
test_tra = {int(s.frame): referans_oku(s) for s in test_kareleri.itertuples()}
test_referans = {k: len(np.unique(tra[tra > 0])) for k, tra in test_tra.items()}

# ---- 3) Kilitli tahminlerle karşılaştır (tahminler yeniden hesaplanmaz) ----
kilitli = pd.read_csv(KILITLI_YOLU)
test_sonuc = pd.DataFrame({
    "kare": [f"t{k:03d}" for k in kilitli.frame],
    "test_block": kilitli.test_block,
    "gercek": [test_referans[k] for k in kilitli.frame],
    "baseline_tahmin": kilitli.baseline_tahmin,
    "nihai_tahmin": kilitli.nihai_tahmin,
})
test_sonuc["baseline_mutlak_hata"] = (test_sonuc.baseline_tahmin - test_sonuc.gercek).abs()
test_sonuc["nihai_mutlak_hata"] = (test_sonuc.nihai_tahmin - test_sonuc.gercek).abs()
print("Tablo 4 - Test kareleri: gerçek ve tahmin edilen hücre sayıları")
print(test_sonuc.to_string(index=False))

ozet = pd.DataFrame([
    {"yontem": "Baseline", **metrikler(test_sonuc.baseline_tahmin, test_sonuc.gercek),
     "ort_sure_ms": kilitli.baseline_sure_ms.mean()},
    {"yontem": f"Nihai pipeline ({NIHAI_AD})", **metrikler(test_sonuc.nihai_tahmin, test_sonuc.gercek),
     "ort_sure_ms": kilitli.nihai_sure_ms.mean()},
]).rename(columns={"Exact": "Exact_count_accuracy"})
print("\nTablo 5 - Test performansı (15 kare)")
print(ozet.round(3).to_string(index=False))

# ---- 4) Tanı ölçütleri ve geliştirme / doğrulama / test karşılaştırması ----
# Etiketler yalnızca analiz için yeniden üretilir; sayımlar kilitli dosyayla aynı olmalıdır.
toplam = Counter()
for s in test_kareleri.itertuples():
    etiket = pipeline_etiket(goruntu_oku(VERI / s.image_file), NIHAI_CFG)
    assert etiket_say(etiket) == int(kilitli.set_index("frame").nihai_tahmin[s.frame])
    toplam.update(tani_hesapla(etiket, test_tra[int(s.frame)]))
test_tani = {a: b / len(test_kareleri) for a, b in toplam.items()}

nihai_ozet = ozet.iloc[1]
karsilastirma = pd.DataFrame({
    "Geliştirme (10)": [dev_nihai["MAE"], dev_nihai["RMSE"], dev_nihai["Bias"]],
    "Doğrulama (5)": [val_nihai.MAE, val_nihai.RMSE, val_nihai.Bias],
    "Test (15)": [nihai_ozet.MAE, nihai_ozet.RMSE, nihai_ozet.Bias],
}, index=["MAE", "RMSE", "Bias"])
print(f"\n{NIHAI_AD}: geliştirme / doğrulama / test")
print(karsilastirma.round(2).to_string())
print("\nTest tanı ölçütleri (kare başına):", {a: round(v, 2) for a, v in test_tani.items()})
print("Blok bazında nihai MAE:", test_sonuc.groupby("test_block").nihai_mutlak_hata.mean().round(2).to_dict(),
      "| baseline MAE:", test_sonuc.groupby("test_block").baseline_mutlak_hata.mean().round(2).to_dict())

test_sonuc.to_csv("test_results.csv", index=False)
ozet.round(3).to_csv("test_summary.csv", index=False)
print("\ntest_results.csv ve test_summary.csv yazıldı.")

In [ ]:
# Bölüm 13 (b): 15 test karesinin tamamının görsel incelemesi (nihai pipeline)
fig, ax = plt.subplots(3, 5, figsize=(25, 17.5), constrained_layout=True)
sonuc_k = test_sonuc.set_index("kare")
for a, s in zip(ax.ravel(), test_kareleri.itertuples()):
    k = int(s.frame)
    g = goruntu_oku(VERI / s.image_file)
    etiket = pipeline_etiket(g, NIHAI_CFG)
    tani = tani_hesapla(etiket, test_tra[k])
    r = sonuc_k.loc[f"t{k:03d}"]
    gorsel = np.dstack([gosterim_icin(g)] * 3)
    gorsel[sinir(etiket)] = [0, 1, 0]
    gorsel[find_boundaries(test_tra[k], mode="outer")] = [1, 0, 0]
    a.imshow(gorsel)
    a.set_title(f"Blok {s.test_block} - t{k:03d}: nihai {r.nihai_tahmin} / ref {r.gercek} (baseline {r.baseline_tahmin})\n"
                f"kaçırılan {tani['kacirilan']}, bölünme {tani['bolunme']}, "
                f"birleşme {tani['birlesme']}, fazla {tani['fazla']}", fontsize=10.5)
    a.axis("off")
fig.suptitle(f"{NIHAI_AD} - 15 test karesi, Seq 02 (yeşil: nesne sınırı, kırmızı: referans işareti)", fontsize=15)
plt.show()

In [ ]:
# Bölüm 13 (c): Hata analizi - kaçırılan hücreler neden kaçırıldı? (yalnızca analiz; yöntem DEĞİŞTİRİLMEZ)
# Her referans hücrenin parlaklığı: Gaussian 15x15 sonrası, TRA işareti içindeki ortalama değer
# (nihai pipeline'ın eşiği bu görüntüye uyguladığı için doğrudan karşılaştırılabilir).

def hucre_tablosu(goruntuler, tralar, kume_adi):
    satirlar = []
    for k, g in goruntuler.items():
        f = cv2.GaussianBlur(g, (NIHAI_CFG["filtre_boyut"], NIHAI_CFG["filtre_boyut"]), 0)
        etiket = pipeline_etiket(g, NIHAI_CFG)
        for h in np.unique(tralar[k][tralar[k] > 0]):
            m = tralar[k] == h
            satirlar.append({"kume": kume_adi, "kare": k, "parlaklik": float(f[m].mean()),
                             "bulundu": bool((etiket[m] > 0).any())})
    return satirlar

test_goruntuler = {int(s.frame): goruntu_oku(VERI / s.image_file) for s in test_kareleri.itertuples()}
hucre_df = pd.DataFrame(hucre_tablosu(dev_goruntuler, dev_tra, "Geliştirme (Seq 01)")
                        + hucre_tablosu(val_goruntuler, val_tra, "Doğrulama (Seq 01)")
                        + hucre_tablosu(test_goruntuler, test_tra, "Test (Seq 02)"))

T = NIHAI_CFG["T"]
analiz = hucre_df.groupby("kume", sort=False).agg(
    hucre_sayisi=("parlaklik", "size"),
    parlaklik_medyan=("parlaklik", "median"),
    T_alti_oran=("parlaklik", lambda p: (p < T).mean()),
    kacirilan_oran=("bulundu", lambda b: 1 - b.mean()),
    kacirilan_parlaklik_maks=("parlaklik", lambda p: p[~hucre_df.loc[p.index, "bulundu"]].max()),
)
analiz["goruntu_p99"] = [np.mean([np.percentile(g, 99) for g in gs.values()])
                         for gs in [dev_goruntuler, val_goruntuler, test_goruntuler]]
print(f"Hücre parlaklığı ve kaçırılma (eşik T={T}):")
print(analiz.round(2).to_string())

fig, ax = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
kutular = np.arange(0, 80, 2)
for kume_adi, renk in [("Geliştirme (Seq 01)", "tab:blue"), ("Test (Seq 02)", "tab:orange")]:
    p = hucre_df[hucre_df.kume == kume_adi].parlaklik
    ax[0].hist(p, bins=kutular, alpha=0.55, color=renk, density=True, label=f"{kume_adi} (medyan {p.median():.1f})")
ax[0].axvline(T, color="red", lw=2, ls="--", label=f"Nihai eşik T={T}")
ax[0].set_xlabel("Hücre parlaklığı (Gaussian 15x15 sonrası)"); ax[0].set_ylabel("Yoğunluk")
ax[0].set_title("Seq 01 ve Seq 02 hücre parlaklık dağılımı"); ax[0].legend(); ax[0].grid(alpha=0.3)

t = hucre_df[hucre_df.kume == "Test (Seq 02)"]
ax[1].hist([t[t.bulundu].parlaklik, t[~t.bulundu].parlaklik], bins=kutular, stacked=True,
           color=["tab:green", "tab:red"], label=["Bulunan", "Kaçırılan"])
ax[1].axvline(T, color="red", lw=2, ls="--", label=f"T={T}")
ax[1].set_xlabel("Hücre parlaklığı"); ax[1].set_ylabel("Hücre sayısı (15 test karesi)")
ax[1].set_title("Test: bulunan ve kaçırılan hücreler"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

## Bölüm 14 – Ardışık Görüntülerde Zamansal Analiz

Görüntüler bir videonun kareleridir; her test bloğu 5 ardışık kareden oluşan ayrı bir zaman segmentidir (blok 1: t042–046, blok 2: t062–066, blok 3: t086–090). Bireysel hücre takibi yapılmaz; analiz kare bazındaki sayımlar üzerinedir. **Kilitli tahminler değiştirilmez.**

**14 (a) – Sayım değişimleri (doküman 14.1–14.5)**
- Her blok için kare numarasına karşı hücre sayısı: gerçek, baseline ve nihai (rapor Şekil 5).
- Ardışık kareler arasındaki değişim (Δ). Referanstaki her değişimin nedeni `man_track.txt` soy ağacından okunur: bölünme, görüş alanına giren ya da çıkan hücre.
- Kararlılık ölçütleri: ortalama |Δ|, en büyük |Δ| ve yön uyumu (tahminin değişim yönü referansla aynı mı?).

**14 (b) – Hatalar zamanda nasıl davranıyor?**
- TRA etiketleri zaman içinde aynı hücreye aynı numarayı verir. Böylece hangi hücrenin hangi karede kaçırıldığı izlenebilir: aynı hücreler ardışık karelerde sürekli mi kaçırılıyor?
- Ani tahmin sıçramalarının nedeni: kare bazında tanı ölçütleri.

**14 (c) – Zamansal karar stratejisi**
- **Önsel bilgi (yalnızca Seq 01):** Ardışık iki kare arasında gerçek hücre sayısı en fazla ne kadar değişebilir? Seq 01'in soy ağacından hesaplanır (`D_MAX`).
- **Kural 1 – Tutarlılık kontrolü:** |Δ tahmin| > `D_MAX` olan geçişler "şüpheli" olarak işaretlenir.
- **Kural 2 – Zamansal medyan:** Bir karenin sayımı, kendisi ve komşu karelerin (pencere 3) medyanı ile düzeltilir; tek karelik sahte sıçramaları bastırır.
- Strateji önce ardışık geliştirme karelerinde (t043–t046) gösterilir, sonra test bloklarına **ayrı bir analiz** olarak uygulanır. Parametreleri (`D_MAX`, pencere 3) testten önce Seq 01'den belirlenmiştir ve kilitli tahminlerin yerine geçmez.

In [ ]:
# Bölüm 14 (a): Bloklar boyunca sayım değişimleri (Şekil 5) ve referans değişimlerinin nedenleri

def soy_agaci_oku(sekans):
    """man_track.txt: her satır = (hücre, ilk kare, son kare, ana hücre)."""
    return pd.read_csv(VERI / f"{sekans}_GT" / "TRA" / "man_track.txt", sep=r"\s+", header=None,
                       names=["hucre", "bas", "son", "ana"])

soy02 = soy_agaci_oku("02")

def gecis_olaylari(soy, tra_t, tra_t1):
    """t -> t+1 geçişinde referanstaki değişimin nedenleri (TRA etiket kümeleri + soy ağacı)."""
    once, sonra = set(np.unique(tra_t)) - {0}, set(np.unique(tra_t1)) - {0}
    yeni, kaybolan = sonra - once, once - sonra
    ana = soy.set_index("hucre").ana
    bolunen = {int(ana[h]) for h in yeni if ana[h] in kaybolan}
    yavru = {h for h in yeni if ana[h] in bolunen}
    beliren = {h for h in yeni - yavru if ana[h] > 0}          # ana hücre önceki karelerde: boşluktan sonra yeniden görünme
    return {"bölünme": len(bolunen), "yavru": len(yavru), "yeni_giren": len(yeni - yavru - beliren),
            "yeniden_beliren": len(beliren), "kaybolan": len(kaybolan - bolunen)}

zaman = test_sonuc.assign(frame=kilitli.frame)
gecisler = []
for blok, b in zaman.groupby("test_block"):
    b = b.reset_index(drop=True)
    for i in range(len(b) - 1):
        t = int(b.frame[i])
        d_ref = int(b.gercek[i + 1] - b.gercek[i])
        d_base = int(b.baseline_tahmin[i + 1] - b.baseline_tahmin[i])
        d_nihai = int(b.nihai_tahmin[i + 1] - b.nihai_tahmin[i])
        olay = gecis_olaylari(soy02, test_tra[t], test_tra[t + 1])
        gecisler.append({"blok": blok, "geçiş": f"t{t:03d}->t{t + 1:03d}", "Δref": d_ref,
                         "Δbaseline": d_base, "Δnihai": d_nihai, **olay})
gecisler = pd.DataFrame(gecisler)
print("Ardışık kareler arasındaki sayım değişimleri ve referans değişiminin nedenleri (soy ağacı):")
print(gecisler.to_string(index=False))
print("(Kontrol: Δref = yavru - bölünen + yeni_giren + yeniden_beliren - kaybolan:",
      bool(((gecisler.yavru - gecisler["bölünme"] + gecisler.yeni_giren + gecisler.yeniden_beliren
             - gecisler.kaybolan) == gecisler["Δref"]).all()), ")")

def kararlilik(g, sutun):
    d, r = g[sutun], g["Δref"]
    return pd.Series({"ort_|Δ|": d.abs().mean(), "maks_|Δ|": d.abs().max(),
                      "yön_uyumu": (np.sign(d) == np.sign(r)).mean(),
                      "ort_|Δ-Δref|": (d - r).abs().mean()})

kararlilik_tablo = pd.concat({ad: gecisler.groupby("blok").apply(lambda g, s=s: kararlilik(g, s))
                              for ad, s in [("Referans", "Δref"), ("Baseline", "Δbaseline"), ("Nihai", "Δnihai")]})
print("\nBlok bazında zamansal kararlılık:")
print(kararlilik_tablo.round(2).to_string())

# Şekil 5
fig, ax = plt.subplots(1, 3, figsize=(20, 5.2), sharey=True, constrained_layout=True)
for a, (blok, b) in zip(ax, zaman.groupby("test_block")):
    a.plot(b.frame, b.gercek, "k*-", ms=13, lw=2.2, label="Gerçek (TRA)")
    a.plot(b.frame, b.baseline_tahmin, "x--", color="tab:purple", ms=9, label="Baseline")
    a.plot(b.frame, b.nihai_tahmin, "o-", color="tab:red", ms=8, lw=2, label=f"Nihai ({NIHAI_AD})")
    for _, r in gecisler[gecisler.blok == blok].iterrows():
        t = int(r["geçiş"][1:4])
        notlar = [f"{v} {k}" for k, v in [("bölünme", r["bölünme"]), ("giren", r["yeni_giren"]),
                                            ("beliren", r["yeniden_beliren"]), ("kaybolan", r["kaybolan"])] if v]
        if notlar:
            a.annotate("\n".join(notlar), (t + 0.5, max(b.gercek) + 0.6), ha="center", fontsize=8.5, color="dimgray")
    b_mae = (b.nihai_tahmin - b.gercek).abs().mean()
    a.set_title(f"Blok {blok} (t{b.frame.min():03d}–t{b.frame.max():03d}) | nihai MAE {b_mae:.1f}, "
                f"baseline MAE {(b.baseline_tahmin - b.gercek).abs().mean():.1f}")
    a.set_xlabel("Kare"); a.set_xticks(b.frame); a.grid(alpha=0.3)
ax[0].set_ylabel("Hücre sayısı"); ax[0].legend(loc="upper left")
fig.suptitle("Şekil 5 - Üç test bloğunda gerçek ve tahmin edilen hücre sayısının zamansal değişimi", fontsize=13)
plt.show()

In [ ]:
# Bölüm 14 (b): Hatalar zamanda nasıl davranıyor? (hücre kimliği bazında kaçırılma + sıçramaların nedeni)
import matplotlib
kare_tani, durum = [], {}
for s in test_kareleri.itertuples():
    k = int(s.frame)
    etiket = pipeline_etiket(test_goruntuler[k], NIHAI_CFG)
    tani = tani_hesapla(etiket, test_tra[k])
    kare_tani.append({"blok": s.test_block, "kare": f"t{k:03d}", "gercek": test_referans[k],
                      "nihai": etiket_say(etiket), **tani})
    for h in np.unique(test_tra[k][test_tra[k] > 0]):
        durum[(int(h), k)] = bool((etiket[test_tra[k] == h] > 0).any())
kare_tani = pd.DataFrame(kare_tani)
kare_tani["Δnihai"] = kare_tani.groupby("blok").nihai.diff()
print("Kare bazında tanı ölçütleri (nihai pipeline):")
print(kare_tani.to_string(index=False))

# Kaçırılan hücrelerin kalıcılığı: t'de kaçırılan hücre t+1'de de var ise yine kaçırılıyor mu?
kalici, toplam = 0, 0
for (h, k), bulundu in durum.items():
    if not bulundu and (h, k + 1) in durum:
        toplam += 1
        kalici += not durum[(h, k + 1)]
print(f"\nBir karede kaçırılıp sonraki karede de görünen {toplam} hücreden {kalici} tanesi (%{100 * kalici / toplam:.0f}) "
      f"sonraki karede de kaçırıldı -> hata sistematik.")

# Isı haritası: en az bir kez kaçırılan hücreler x 15 test karesi
kareler = [int(f) for f in kilitli.frame]
hic_kacan = sorted({h for (h, k), b in durum.items() if not b})
harita = np.full((len(hic_kacan), len(kareler)), np.nan)
for i, h in enumerate(hic_kacan):
    for j, k in enumerate(kareler):
        if (h, k) in durum:
            harita[i, j] = 1.0 if durum[(h, k)] else 0.0
fig, ax = plt.subplots(figsize=(12, 0.32 * len(hic_kacan) + 2), constrained_layout=True)
ax.imshow(np.ma.masked_invalid(harita), cmap=matplotlib.colors.ListedColormap(["tab:red", "tab:green"]),
          aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(kareler)), [f"t{k:03d}" for k in kareler], rotation=45)
ax.set_yticks(range(len(hic_kacan)), [f"hücre {h}" for h in hic_kacan], fontsize=8)
for x in [4.5, 9.5]:
    ax.axvline(x, color="black", lw=2)
ax.set_title("En az bir kez kaçırılan hücreler (kırmızı: kaçırıldı, yeşil: bulundu, beyaz: karede yok)")
plt.show()

In [ ]:
# Bölüm 14 (c): Zamansal karar stratejisi (kilitli tahminleri DEĞİŞTİRMEZ; ayrı analiz)

# ---- Önsel bilgi: yalnızca Seq 01 soy ağacından ----
soy01 = soy_agaci_oku("01")
kare_sayisi01 = int(soy01.son.max()) + 1
sayim01 = np.array([((soy01.bas <= t) & (soy01.son >= t)).sum() for t in range(kare_sayisi01)])
d01 = np.diff(sayim01)
D_MAX = int(np.abs(d01).max())
print(f"Seq 01 ({kare_sayisi01} kare): ardışık karelerde gerçek sayım değişimi dağılımı:",
      pd.Series(d01).value_counts().sort_index().to_dict())
print(f"-> D_MAX = {D_MAX}: iki ardışık kare arasında gerçek sayı en fazla {D_MAX} değişmiştir.\n")

def zamansal_medyan(seri, pencere=3):
    yari = pencere // 2
    return np.array([np.median(seri[max(0, i - yari):i + yari + 1]) for i in range(len(seri))])

def strateji_uygula(kareler, tahmin, gercek):
    tahmin, gercek = np.asarray(tahmin), np.asarray(gercek)
    supheli = [f"t{kareler[i]:03d}->t{kareler[i + 1]:03d}" for i in range(len(tahmin) - 1)
               if abs(tahmin[i + 1] - tahmin[i]) > D_MAX]
    duzeltilmis = zamansal_medyan(tahmin)
    return {"şüpheli_geçişler": supheli or "-",
            "ham_MAE": np.abs(tahmin - gercek).mean(), "medyan_MAE": np.abs(duzeltilmis - gercek).mean(),
            "ham_ort_|Δ|": np.abs(np.diff(tahmin)).mean(), "medyan_ort_|Δ|": np.abs(np.diff(duzeltilmis)).mean(),
            "ref_ort_|Δ|": np.abs(np.diff(gercek)).mean()}

# ---- 1) Ardışık geliştirme kareleri (t043-t046): stratejinin testten önce gösterimi ----
ardisik_dev = [43, 44, 45, 46]
satirlar14 = [{"segment": "Geliştirme t043-t046", "yontem": ad,
               **strateji_uygula(ardisik_dev, [say(dev_goruntuler[k]) for k in ardisik_dev],
                                 [dev_referans[k] for k in ardisik_dev])}
              for ad, say in [("Baseline", lambda g: baseline_segment(g, BASELINE_MIN_ALAN)[0]),
                              ("Nihai", lambda g: pipeline_say(g, NIHAI_CFG))]]

# ---- 2) Test blokları (kilitli tahminler üzerinde analiz) ----
for blok, b in zaman.groupby("test_block"):
    for ad, sutun in [("Baseline", "baseline_tahmin"), ("Nihai", "nihai_tahmin")]:
        satirlar14.append({"segment": f"Test blok {blok}", "yontem": ad,
                           **strateji_uygula(list(b.frame), b[sutun], b.gercek)})
strateji_tablo = pd.DataFrame(satirlar14)
print("Zamansal strateji: şüpheli geçişler ve zamansal medyanın etkisi")
print(strateji_tablo.round(2).to_string(index=False))

test_str = strateji_tablo[strateji_tablo.segment.str.startswith("Test")]
for ad in ["Baseline", "Nihai"]:
    t = test_str[test_str.yontem == ad]
    print(f"\n{ad} - 15 test karesi: ham MAE {t.ham_MAE.mean():.2f} -> zamansal medyan MAE {t.medyan_MAE.mean():.2f} | "
          f"ort |Δ| {t['ham_ort_|Δ|'].mean():.2f} -> {t['medyan_ort_|Δ|'].mean():.2f} (referans {t['ref_ort_|Δ|'].mean():.2f})")

In [ ]:
# Bölüm 14 (d): Başarılı, hatalı ve zorlayıcı örnekler (rapor Şekil 6) - bölgeler nesnel ölçütlerle seçilir
PENCERE = 260

def hucre_durumlari(etiket, tra):
    """Her referans hücre: merkez, kaç nesneyle örtüşüyor (0 = kaçırıldı, >1 = bölündü)."""
    out = []
    for h in np.unique(tra[tra > 0]):
        m = tra == h
        cy, cx = np.argwhere(m).mean(axis=0)
        out.append({"hucre": int(h), "cy": cy, "cx": cx, "nesne": len(np.unique(etiket[m & (etiket > 0)]))})
    return pd.DataFrame(out)

def pencere_skoru(d, cy, cx):
    icerde = d[(abs(d.cy - cy) < PENCERE / 2) & (abs(d.cx - cx) < PENCERE / 2)]
    return (icerde.nesne == 1).sum(), (icerde.nesne == 0).sum(), (icerde.nesne > 1).sum()

adaylar = []
for k in kareler:
    etiket = pipeline_etiket(test_goruntuler[k], NIHAI_CFG)
    d = hucre_durumlari(etiket, test_tra[k])
    for r in d.itertuples():
        dogru, kacan, bolunen = pencere_skoru(d, r.cy, r.cx)
        adaylar.append({"kare": k, "cy": r.cy, "cx": r.cx, "dogru": dogru, "kacan": kacan, "bolunen": bolunen})
adaylar = pd.DataFrame(adaylar)

ornekler = {
    "Başarılı: en çok hücrenin hatasız bulunduğu bölge": adaylar[(adaylar.kacan == 0) & (adaylar.bolunen == 0)]
        .sort_values("dogru", ascending=False).iloc[0],
    "Hatalı: en çok hücrenin kaçırıldığı bölge": adaylar.sort_values(["kacan", "dogru"], ascending=False).iloc[0],
    "Zorlayıcı: bir hücrenin yanlışlıkla bölündüğü bölge": adaylar[adaylar.bolunen > 0]
        .sort_values("dogru", ascending=False).iloc[0],
}

fig, ax = plt.subplots(2, 3, figsize=(20, 13.5), constrained_layout=True)
for j, (baslik, r) in enumerate(ornekler.items()):
    k = int(r.kare)
    g = test_goruntuler[k]
    y0 = int(np.clip(r.cy - PENCERE / 2, 0, g.shape[0] - PENCERE)); x0 = int(np.clip(r.cx - PENCERE / 2, 0, g.shape[1] - PENCERE))
    bolge = (slice(y0, y0 + PENCERE), slice(x0, x0 + PENCERE))
    etiket = pipeline_etiket(g, NIHAI_CFG)
    ax[0, j].imshow(gosterim_icin(g)[bolge], cmap="gray")
    ax[0, j].set_title(f"{baslik}\nt{k:03d}, bölge y={y0}, x={x0} (gösterim için parlatılmış)", fontsize=11)
    gorsel = np.dstack([gosterim_icin(g)] * 3)
    gorsel[sinir(etiket)] = [0, 1, 0]
    gorsel[find_boundaries(test_tra[k], mode="outer")] = [1, 0, 0]
    ax[1, j].imshow(gorsel[bolge])
    ax[1, j].set_title(f"doğru {int(r.dogru)}, kaçırılan {int(r.kacan)}, bölünen {int(r.bolunen)} "
                       f"(yeşil: nesne, kırmızı: referans)", fontsize=11)
for a in ax.ravel():
    a.axis("off")
fig.suptitle(f"Şekil 6 - {NIHAI_AD}: başarılı, hatalı ve zorlayıcı örnekler (test kareleri)", fontsize=14)
plt.show()

## Bölüm 15 – Deneysel Sonuçların Tartışılması

Dokümanın 15. bölümündeki 9 soru raporun 7. bölümünde cevaplanır. Notebook'ta bu cevapları destekleyen iki özet görsel üretilir:

- **15 (a) – Şekil 1:** Nihai pipeline'ın akış diyagramı (her aşama, parametresi ve gerekçesiyle).
- **15 (b) – İyileşme zinciri:** Baseline'dan nihai yönteme giden yolda hangi değişikliğin geliştirme MAE'sini ne kadar değiştirdiği (soru 5), ve nihai yöntemin geliştirme / doğrulama / test performansı (soru 6–7).

Bu bölümde yeni bir deney yapılmaz; tüm sayılar önceki bölümlerde hesaplanan değişkenlerden okunur.

In [ ]:
# Bölüm 15 (a): Şekil 1 - nihai pipeline akış diyagramı (parametreler final_pipeline.json'dan okunur)
from matplotlib.patches import FancyBboxPatch

p = json.loads(JSON_YOLU.read_text(encoding="utf-8"))["parametreler"]
adimlar = [
    ("Ham görüntü", "1024x1024, uint8\n(Seq 01 / Seq 02)", "#dde3ea"),
    ("Gaussian filtre", f"{p['filtre_boyut']}x{p['filtre_boyut']} çekirdek\ngürültü ve çekirdekçik\ndokusunu yumuşatır", "#cfe2f3"),
    ("Sabit eşik", f"piksel > T = {p['T']}\nhücre / arka plan\n(ikili maske)", "#fde9c9"),
    ("Closing", f"{p['kapama']}x{p['kapama']} elips\nçatlak ve boşlukları\nkapatır", "#d9ead3"),
    ("Delik doldurma", "hücre içi\nsiyah bölgeleri\ndoldurur", "#d9ead3"),
    ("Distance transform\n+ watershed", f"min_distance = {p['min_distance']}\nbirbirine değen\nhücreleri ayırır", "#ead1dc"),
    ("Bağlı bileşen\n+ min. alan", f"8-komşuluk\nalan < {p['min_alan']} px atılır\n(gürültü lekeleri)", "#ead1dc"),
    ("Hücre sayısı", "kare başına\nnesne sayısı", "#dde3ea"),
]
fig, ax = plt.subplots(figsize=(22, 4.2))
w, h, aralik = 2.3, 1.9, 0.45
for i, (baslik, aciklama, renk) in enumerate(adimlar):
    x = i * (w + aralik)
    ax.add_patch(FancyBboxPatch((x, 0), w, h, boxstyle="round,pad=0.04,rounding_size=0.15", fc=renk, ec="#444", lw=1.4))
    ax.text(x + w / 2, h - 0.38, baslik, ha="center", va="center", fontsize=11.5, fontweight="bold")
    ax.text(x + w / 2, 0.68, aciklama, ha="center", va="center", fontsize=9.5)
    if i < len(adimlar) - 1:
        ax.annotate("", xy=(x + w + aralik - 0.03, h / 2), xytext=(x + w + 0.03, h / 2),
                    arrowprops=dict(arrowstyle="-|>", lw=2, color="#444"))
for ad, x0, x1, renk in [("Ön işleme", 1, 1, "#3d85c6"), ("Segmentasyon", 2, 2, "#e69138"),
                         ("Morfoloji", 3, 4, "#6aa84f"), ("Nesne ayırma ve sayım", 5, 6, "#a64d79")]:
    xa, xb = x0 * (w + aralik), x1 * (w + aralik) + w
    ax.plot([xa, xb], [-0.25, -0.25], color=renk, lw=4)
    ax.text((xa + xb) / 2, -0.5, ad, ha="center", va="center", fontsize=10.5, color=renk, fontweight="bold")
ax.set_xlim(-0.2, len(adimlar) * (w + aralik)); ax.set_ylim(-0.8, h + 0.2); ax.axis("off")
ax.set_title(f"Şekil 1 - Nihai görüntü işleme pipeline'ı ({NIHAI_AD}, final_pipeline.json)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Bölüm 15 (b): Hangi değişiklik ne kadar iyileşme sağladı? (soru 5) + geliştirme / doğrulama / test (soru 6-7)
def dev_mae_of(deney_id):
    return float(next(d for d in DENEYLER if d["deney_id"] == deney_id)["MAE"])

zincirler = {
    "Blok Otsu yolu (Pipeline A)": [("E01", "Baseline\n(Otsu)"), ("E11", "+ Gaussian\n15x15"), ("E21", "Otsu ->\nBlok Otsu 8x8"),
                                    ("E30", "+ closing\n11x11"), ("E35", "+ doldurma\n+ watershed"), ("E36", "+ min. alan\n100")],
    "Sabit eşik yolu (Pipeline D, nihai)": [("E01", "Baseline\n(Otsu)"), ("E11", "+ Gaussian\n15x15"), ("E17", "Otsu ->\nsabit T=7"),
                                            ("E42", "+ closing, doldurma,\nwatershed, min. alan")],
}
fig, ax = plt.subplots(1, 3, figsize=(24, 5.6), gridspec_kw={"width_ratios": [1.5, 1, 0.9]}, constrained_layout=True)
for a, (ad, zincir) in zip(ax[:2], zincirler.items()):
    degerler = [dev_mae_of(d) for d, _ in zincir]
    renkler = ["tab:gray"] + ["tab:green" if round(b, 2) < round(o, 2) else "tab:red" if round(b, 2) > round(o, 2)
                              else "tab:blue" for o, b in zip(degerler, degerler[1:])]
    a.bar(range(len(zincir)), degerler, color=renkler)
    for i, ((d, _), v) in enumerate(zip(zincir, degerler)):
        a.text(i, v + 0.08, f"{v:.1f}\n({d})", ha="center", fontsize=9.5)
    a.set_xticks(range(len(zincir)), [e for _, e in zincir], fontsize=9.5)
    a.set_ylabel("Geliştirme MAE"); a.set_title(f"{ad}\n(yeşil: iyileşme, kırmızı: kötüleşme, mavi: değişim yok)"); a.grid(alpha=0.3, axis="y")
    a.set_ylim(0, max(degerler) * 1.25)

kumeler = ["Geliştirme\n(10, Seq 01)", "Doğrulama\n(5, Seq 01)", "Test\n(15, Seq 02)"]
base_vals = [dev_mae_of("E01"), dogrulama.set_index("pipeline").loc["Baseline"].MAE, ozet.iloc[0].MAE]
nihai_vals = [dev_mae_of(NIHAI_DENEY), val_nihai.MAE, ozet.iloc[1].MAE]
x = np.arange(3)
ax[2].bar(x - 0.2, base_vals, 0.4, color="tab:purple", label="Baseline")
ax[2].bar(x + 0.2, nihai_vals, 0.4, color="tab:red", label=f"Nihai ({NIHAI_AD})")
for i in range(3):
    ax[2].text(i - 0.2, base_vals[i] + 0.1, f"{base_vals[i]:.1f}", ha="center")
    ax[2].text(i + 0.2, nihai_vals[i] + 0.1, f"{nihai_vals[i]:.1f}", ha="center")
ax[2].set_xticks(x, kumeler); ax[2].set_ylabel("MAE"); ax[2].legend(); ax[2].grid(alpha=0.3, axis="y")
ax[2].set_title("Baseline ve nihai: geliştirme / doğrulama / test")
plt.show()

print("Özet (MAE):")
print(pd.DataFrame({"Baseline": base_vals, f"Nihai ({NIHAI_AD})": nihai_vals},
                   index=["Geliştirme", "Doğrulama", "Test"]).round(2).to_string())
